# ANA_EDU_FIN_CLI V15 — Tuning de Produção Homologado no Sandbox

Esta versão consolida as correções funcionais e as otimizações comprovadas na execução completa da V14.

### Diretrizes da V15
- **Carga Hive correta no sandbox:** sem `saveAsTable`; gravação por `INSERT OVERWRITE` da competência e validação pós-carga obrigatória.
- **Idempotência por `DT_MES_EXEA`:** reexecuções substituem somente a competência corrente, preservando as demais partições.
- **Validações realmente bloqueantes:** nenhuma criação/carga é permitida enquanto chave, cardinalidade e calendário não estiverem aprovados.
- **Spark dimensionado explicitamente na primeira célula:** 4 a 12 executores dinâmicos, alvo inicial de 8, 4 cores e 4 GB por executor.
- **JDBC controlado em 16 fluxos:** preserva a pressão já validada sobre o DB2.
- **Público sem segunda leitura JDBC:** a redução por conta é materializada uma vez e reutilizada pelos dois ramos do cálculo.
- **Envelope mensal exato:** primeiro e último mês são cortados pelos limites reais `MIN(DT_REF_INI)` e `MAX(DT_REF_FIM)`.
- **Fim do acumulador recursivo:** cada mês é agregado/materializado uma vez; a consolidação histórica executa um único `UNION ALL + GROUP BY` final.
- **RDPR e renda preservados:** a arquitetura que demonstrou partition pruning e baixo tempo permanece intacta.
- **Diagnósticos descartáveis e fail-open:** células `PERF_DIAGNOSTICO_REMOVER` não são pré-requisito funcional do ETL.


## Mapa de execução V15

```text
[1. Configuração Spark explícita — primeira célula]
                   │
[2. Metadados de categorias e mapa pequeno em broadcast]
                   │
[3. Público DB2 — 16 JDBC → redução por conta materializada UMA VEZ]
                   │                   └──► ciclo CT_GRDR_FNCO (16 JDBC)
                   ▼
[4. Janela financeira materializada]
                   │
[5. Perfil financeiro DB2 — 16 JDBC]
                   │
[6. RDPR Hive com partition pruning → renda presumida]
                   │
[7. Envelope exato MIN/MAX da janela]
                   │
[8. Meses ativos — cada mês: 16 JDBC → agrega → materializa uma vez]
                   │
[9. UNION dos meses materializados → GROUP BY CD_CLI UMA VEZ]
                   │
[10. Camada analítica modular → DataFrame final 72 colunas]
                   │
[11. Qualidade + calendário — GATE BLOQUEANTE]
                   │
[12. DDL Hive/Parquet no sandbox]
                   │
[13. INSERT OVERWRITE da partição DT_MES_EXEA]
                   │
[14. COUNT pós-carga = qt_final → somente então CARGA_OK]
```


> **V15 — objetivo:** corrigir a persistência no sandbox e reduzir o caminho crítico comprovado pela V13/V14. A execução anterior mostrou que os meses/acumuladores concentravam a maior parte do tempo; por isso esta versão elimina reagrupamentos cumulativos e mantém o DB2 limitado a 16 fluxos simultâneos por leitura.


## Perfil de recursos V15 — homologação no sandbox

A V15 mantém `AMBIENTE=MODELAGEM` para validar a tabela em `sbx_t2i2016`, mas usa um patamar de recursos compatível com o estudo do ambiente de produção: **8 executores iniciais, elasticidade de 4 a 12, 4 cores e 4 GB por executor, até 48 cores Spark e 480 partições iniciais de shuffle**. O paralelismo JDBC permanece limitado a **16 fluxos**, evitando aumentar a pressão sobre o DB2.

A promoção futura para `AMBIENTE=PRODUCAO` deve alterar somente o contexto operacional/target, não espalhar configurações Spark pelo notebook.


## 1. Configuração Spark e conexão local

Todas as propriedades Spark são aplicadas nesta primeira célula executável, antes da criação da sessão.


In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    # V15: homologação no sandbox/modelagem com perfil de recursos inspirado no ambiente de produção.
    # REGRA: todas as configurações Spark permanecem centralizadas exclusivamente nesta célula.
    gerenciador_spark = GerenciadorLocal(
        nome_sessao="ana-edu-fin-cli-v15-prod-tuning",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            "SANDBOX": "t2i2016",
            "AMBIENTE": "MODELAGEM",
        },
        nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        # Driver reduzido em relação à V14: o pipeline não coleta grandes volumes no driver.
        driver_memory="12g",
        # Com dynamic allocation, este valor define o patamar inicial solicitado.
        num_executors=8,
        executor_memory="4g",
        executor_cores=4,
        jars=[
            "/dados/shared/bin/ojdbc8.jar",
        ],
        spark_conf={
            # -----------------------------------------------------------------
            # Recursos / elasticidade
            # -----------------------------------------------------------------
            "spark.driver.memoryOverhead": "4g",
            "spark.executor.memoryOverhead": "2g",
            "spark.dynamicAllocation.enabled": "true",
            "spark.dynamicAllocation.minExecutors": "4",
            "spark.dynamicAllocation.initialExecutors": "8",
            "spark.dynamicAllocation.maxExecutors": "12",

            # -----------------------------------------------------------------
            # Serialização
            # -----------------------------------------------------------------
            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",

            # -----------------------------------------------------------------
            # AQE / shuffle
            # 480 mantém ~10 partições iniciais por core no teto de 48 cores.
            # O AQE pode coalescer partições menores após observar o volume real.
            # -----------------------------------------------------------------
            "spark.speculation": "false",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
            "spark.sql.adaptive.advisoryPartitionSizeInBytes": "64m",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "480",

            # -----------------------------------------------------------------
            # Broadcast
            # Mantido desligado globalmente. Estruturas comprovadamente pequenas
            # (ex.: mapa de categorias) usam hint BROADCAST explícito.
            # -----------------------------------------------------------------
            "spark.sql.autoBroadcastJoinThreshold": "-1",
            "spark.sql.broadcastTimeout": "8000",

            # -----------------------------------------------------------------
            # I/O / estabilidade
            # JDBC permanece controlado em até 16 fluxos no código funcional.
            # -----------------------------------------------------------------
            "spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive": "true",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",

            # A carga final usa INSERT OVERWRITE estático da competência.
            # Mantido centralizado para compatibilidade com operações particionadas.
            "spark.sql.sources.partitionOverwriteMode": "dynamic",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


## 2. Conexão remota


In [ ]:
try:
    %run ./src/utils/gerenciador_spark_v2.ipynb
    %run ./src/utils/gerenciador_db2_spark_v2.ipynb
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


## 3. Contexto Spark e cliente DB2


In [ ]:
%%spark

import os

ambiente = obter_variavel_ambiente("AMBIENTE").upper()
dominio = obter_variavel_ambiente("DOMINIO").lower()
hoje = obter_variavel_ambiente("HOJE")

if ambiente == "MODELAGEM":
    sandbox = obter_variavel_ambiente("SANDBOX").lower()
    database = f"sbx_{sandbox}"
else:
    database = f"hive_{dominio}"

env_spark = dict(os.environ)

conector_db2 = criar_conector_db2_spark(env=env_spark)


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Contexto real da sessão Spark — somente metadados, sem job
try:
    print("\n[PERF-DIAG] CONTEXTO SPARK")
    print("defaultParallelism:", spark.sparkContext.defaultParallelism)
    print("executores visíveis (inclui driver conforme backend):", spark.sparkContext._jsc.sc().getExecutorMemoryStatus().size())
    print("spark.executor.instances:", spark.sparkContext.getConf().get("spark.executor.instances", "NAO_DEFINIDO"))
    print("spark.executor.cores:", spark.sparkContext.getConf().get("spark.executor.cores", "NAO_DEFINIDO"))
    print("spark.executor.memory:", spark.sparkContext.getConf().get("spark.executor.memory", "NAO_DEFINIDO"))
    print("spark.driver.memory:", spark.sparkContext.getConf().get("spark.driver.memory", "NAO_DEFINIDO"))
    print("spark.driver.memoryOverhead:", spark.sparkContext.getConf().get("spark.driver.memoryOverhead", "NAO_DEFINIDO"))
    print("spark.executor.memoryOverhead:", spark.sparkContext.getConf().get("spark.executor.memoryOverhead", "NAO_DEFINIDO"))
    print("spark.dynamicAllocation.enabled:", spark.sparkContext.getConf().get("spark.dynamicAllocation.enabled", "NAO_DEFINIDO"))
    print("spark.dynamicAllocation.minExecutors:", spark.sparkContext.getConf().get("spark.dynamicAllocation.minExecutors", "NAO_DEFINIDO"))
    print("spark.dynamicAllocation.initialExecutors:", spark.sparkContext.getConf().get("spark.dynamicAllocation.initialExecutors", "NAO_DEFINIDO"))
    print("spark.dynamicAllocation.maxExecutors:", spark.sparkContext.getConf().get("spark.dynamicAllocation.maxExecutors", "NAO_DEFINIDO"))
    print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
    print("spark.sql.adaptive.enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
    print("spark.sql.adaptive.coalescePartitions.enabled:", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
    print("spark.sql.adaptive.advisoryPartitionSizeInBytes:", spark.conf.get("spark.sql.adaptive.advisoryPartitionSizeInBytes"))
    print("spark.sql.adaptive.skewJoin.enabled:", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
    print("spark.sql.autoBroadcastJoinThreshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


## 4. Parâmetros do usuário

Configurações operacionais expostas da execução.


In [ ]:
%%spark

nome_tabela = "ana_edu_fin_cli"
tabela_spark = f"sbx_t2i2016.{nome_tabela}"
periodo = 1
qtd_mes_pbco_alvo = 1

# False preserva histórico e permite overwrite idempotente somente da competência corrente.
# Use True apenas quando houver mudança deliberada de DDL e a recriação integral for desejada.
RECRIAR_TABELA_FINAL = False

# DataFrame final observado na V13 tinha ~2 GB em cache; 32 writers evitam 240 arquivos pequenos.
QTD_PARTICOES_ESCRITA = 32


### 4.1 Dicionário completo de categorias (Single Point of Truth com IN_PARTICIPA_CALCULO)

`CATEGORIAS` é a fonte única oficial para tipo, grupo, descrição, classificação Radar, indicador agro e flag de participação nos cálculos.


In [ ]:
%%spark

# Dicionário Oficial: Governança Unificada com IN_PARTICIPA_CALCULO
CATEGORIAS = {
    0: {
        'TIPO': None,
        'CD_GRUPO': 0,
        'TX_GRUPO': 'Sem categoria',
        'CD_CATEGORIA': 0,
        'TX_CATEGORIA': 'Sem categoria',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 0,
        'TX_CLASS_RADAR': 'Outras Entradas',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    1: {
        'TIPO': 'C',
        'CD_GRUPO': 1,
        'TX_GRUPO': 'Receitas',
        'CD_CATEGORIA': 1,
        'TX_CATEGORIA': 'Salário',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 1,
        'TX_CLASS_RADAR': 'Renda',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    2: {
        'TIPO': 'C',
        'CD_GRUPO': 1,
        'TX_GRUPO': 'Receitas',
        'CD_CATEGORIA': 2,
        'TX_CATEGORIA': 'Vale Alimentação',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 1,
        'TX_CLASS_RADAR': 'Renda',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    3: {
        'TIPO': 'C',
        'CD_GRUPO': 1,
        'TX_GRUPO': 'Receitas',
        'CD_CATEGORIA': 3,
        'TX_CATEGORIA': 'Restituição de IR',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 2,
        'TX_CLASS_RADAR': 'Estorno',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    4: {
        'TIPO': 'C',
        'CD_GRUPO': 1,
        'TX_GRUPO': 'Receitas',
        'CD_CATEGORIA': 4,
        'TX_CATEGORIA': 'Bonificação',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 1,
        'TX_CLASS_RADAR': 'Renda',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    5: {
        'TIPO': 'C',
        'CD_GRUPO': 1,
        'TX_GRUPO': 'Receitas',
        'CD_CATEGORIA': 5,
        'TX_CATEGORIA': 'Outros Rendimentos',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 1,
        'TX_CLASS_RADAR': 'Renda',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    6: {
        'TIPO': 'D',
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
        'CD_CATEGORIA': 6,
        'TX_CATEGORIA': 'Água',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    7: {
        'TIPO': 'D',
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
        'CD_CATEGORIA': 7,
        'TX_CATEGORIA': 'Eletricidade e Gás',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    9: {
        'TIPO': 'D',
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
        'CD_CATEGORIA': 9,
        'TX_CATEGORIA': 'Compra de Imóvel',
        'CD_IR': 2,
        'TX_IR': 'Bens e direitos',
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    10: {
        'TIPO': 'D',
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
        'CD_CATEGORIA': 10,
        'TX_CATEGORIA': 'Aluguel e Condomínio',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    11: {
        'TIPO': 'D',
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
        'CD_CATEGORIA': 11,
        'TX_CATEGORIA': 'Móveis e Utensílios',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    12: {
        'TIPO': 'D',
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
        'CD_CATEGORIA': 12,
        'TX_CATEGORIA': 'Serviços e Manutenção',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    13: {
        'TIPO': 'D',
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
        'CD_CATEGORIA': 13,
        'TX_CATEGORIA': 'Empregados',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    14: {
        'TIPO': 'D',
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
        'CD_CATEGORIA': 14,
        'TX_CATEGORIA': 'Animais e Pets',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    15: {
        'TIPO': 'D',
        'CD_GRUPO': 3,
        'TX_GRUPO': 'Educação',
        'CD_CATEGORIA': 15,
        'TX_CATEGORIA': 'Educação Superior',
        'CD_IR': 1,
        'TX_IR': 'Pagamentos efetuados',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    16: {
        'TIPO': 'D',
        'CD_GRUPO': 3,
        'TX_GRUPO': 'Educação',
        'CD_CATEGORIA': 16,
        'TX_CATEGORIA': 'Colégio',
        'CD_IR': 1,
        'TX_IR': 'Pagamentos efetuados',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    17: {
        'TIPO': 'D',
        'CD_GRUPO': 3,
        'TX_GRUPO': 'Educação',
        'CD_CATEGORIA': 17,
        'TX_CATEGORIA': 'Idiomas',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    18: {
        'TIPO': 'D',
        'CD_GRUPO': 3,
        'TX_GRUPO': 'Educação',
        'CD_CATEGORIA': 18,
        'TX_CATEGORIA': 'Publicações e Papelaria',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    20: {
        'TIPO': 'D',
        'CD_GRUPO': 3,
        'TX_GRUPO': 'Educação',
        'CD_CATEGORIA': 20,
        'TX_CATEGORIA': 'Outros Gastos, Educação',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    21: {
        'TIPO': 'D',
        'CD_GRUPO': 4,
        'TX_GRUPO': 'Lazer',
        'CD_CATEGORIA': 21,
        'TX_CATEGORIA': 'Viagens e Lazer',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    22: {
        'TIPO': 'D',
        'CD_GRUPO': 4,
        'TX_GRUPO': 'Lazer',
        'CD_CATEGORIA': 22,
        'TX_CATEGORIA': 'Esportes e Academia',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    25: {
        'TIPO': 'D',
        'CD_GRUPO': 4,
        'TX_GRUPO': 'Lazer',
        'CD_CATEGORIA': 25,
        'TX_CATEGORIA': 'Cultura e Entretenimento',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    27: {
        'TIPO': 'D',
        'CD_GRUPO': 5,
        'TX_GRUPO': 'Saúde',
        'CD_CATEGORIA': 27,
        'TX_CATEGORIA': 'Plano de Saúde',
        'CD_IR': 1,
        'TX_IR': 'Pagamentos efetuados',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    28: {
        'TIPO': 'D',
        'CD_GRUPO': 5,
        'TX_GRUPO': 'Saúde',
        'CD_CATEGORIA': 28,
        'TX_CATEGORIA': 'Serviços de Saúde',
        'CD_IR': 1,
        'TX_IR': 'Pagamentos efetuados',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    29: {
        'TIPO': 'D',
        'CD_GRUPO': 5,
        'TX_GRUPO': 'Saúde',
        'CD_CATEGORIA': 29,
        'TX_CATEGORIA': 'Dentista',
        'CD_IR': 1,
        'TX_IR': 'Pagamentos efetuados',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    30: {
        'TIPO': 'D',
        'CD_GRUPO': 5,
        'TX_GRUPO': 'Saúde',
        'CD_CATEGORIA': 30,
        'TX_CATEGORIA': 'Farmácias e Drogarias',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    32: {
        'TIPO': 'D',
        'CD_GRUPO': 6,
        'TX_GRUPO': 'Alimentação',
        'CD_CATEGORIA': 32,
        'TX_CATEGORIA': 'Feira e Supermercado',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    35: {
        'TIPO': 'D',
        'CD_GRUPO': 6,
        'TX_GRUPO': 'Alimentação',
        'CD_CATEGORIA': 35,
        'TX_CATEGORIA': 'Bar, Rest. e Padaria',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    36: {
        'TIPO': 'D',
        'CD_GRUPO': 7,
        'TX_GRUPO': 'Transporte',
        'CD_CATEGORIA': 36,
        'TX_CATEGORIA': 'Compra de Veículo',
        'CD_IR': 2,
        'TX_IR': 'Bens e direitos',
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    37: {
        'TIPO': 'D',
        'CD_GRUPO': 7,
        'TX_GRUPO': 'Transporte',
        'CD_CATEGORIA': 37,
        'TX_CATEGORIA': 'Combustível',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    38: {
        'TIPO': 'D',
        'CD_GRUPO': 7,
        'TX_GRUPO': 'Transporte',
        'CD_CATEGORIA': 38,
        'TX_CATEGORIA': 'Estacionamento e Pedágio',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    39: {
        'TIPO': 'D',
        'CD_GRUPO': 7,
        'TX_GRUPO': 'Transporte',
        'CD_CATEGORIA': 39,
        'TX_CATEGORIA': 'Seguro de Veículo',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    40: {
        'TIPO': 'D',
        'CD_GRUPO': 7,
        'TX_GRUPO': 'Transporte',
        'CD_CATEGORIA': 40,
        'TX_CATEGORIA': 'Serviços e Manutenção',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    41: {
        'TIPO': 'D',
        'CD_GRUPO': 7,
        'TX_GRUPO': 'Transporte',
        'CD_CATEGORIA': 41,
        'TX_CATEGORIA': 'Transporte Urbano e Apps',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    42: {
        'TIPO': 'D',
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
        'CD_CATEGORIA': 42,
        'TX_CATEGORIA': 'Vestuário e Acessórios',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    43: {
        'TIPO': 'D',
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
        'CD_CATEGORIA': 43,
        'TX_CATEGORIA': 'Cuidado Pessoal e Beleza',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    44: {
        'TIPO': 'D',
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
        'CD_CATEGORIA': 44,
        'TX_CATEGORIA': 'Compras Diversas',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    45: {
        'TIPO': 'D',
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
        'CD_CATEGORIA': 45,
        'TX_CATEGORIA': 'Pensão Alimentícia',
        'CD_IR': 1,
        'TX_IR': 'Pagamentos efetuados',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    46: {
        'TIPO': 'D',
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
        'CD_CATEGORIA': 46,
        'TX_CATEGORIA': 'Seguros e Previdência',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    47: {
        'TIPO': 'D',
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
        'CD_CATEGORIA': 47,
        'TX_CATEGORIA': 'Doação',
        'CD_IR': 4,
        'TX_IR': 'Doações efetuadas',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    48: {
        'TIPO': 'D',
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
        'CD_CATEGORIA': 48,
        'TX_CATEGORIA': 'Gasto com Familiares',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    49: {
        'TIPO': 'D',
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
        'CD_CATEGORIA': 49,
        'TX_CATEGORIA': 'Presentes',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    51: {
        'TIPO': 'D',
        'CD_GRUPO': 9,
        'TX_GRUPO': 'Comunicação',
        'CD_CATEGORIA': 51,
        'TX_CATEGORIA': 'Telefonia e Internet',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    53: {
        'TIPO': 'D',
        'CD_GRUPO': 9,
        'TX_GRUPO': 'Comunicação',
        'CD_CATEGORIA': 53,
        'TX_CATEGORIA': 'Assinatura TV e Streaming',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    54: {
        'TIPO': 'D',
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
        'CD_CATEGORIA': 54,
        'TX_CATEGORIA': 'IPTU',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    55: {
        'TIPO': 'D',
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
        'CD_CATEGORIA': 55,
        'TX_CATEGORIA': 'IPVA e Gastos Detran',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    56: {
        'TIPO': 'D',
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
        'CD_CATEGORIA': 56,
        'TX_CATEGORIA': 'Imposto de Renda',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    57: {
        'TIPO': 'D',
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
        'CD_CATEGORIA': 57,
        'TX_CATEGORIA': 'ISS(Imposto sobre Serviços)',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    58: {
        'TIPO': 'D',
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
        'CD_CATEGORIA': 58,
        'TX_CATEGORIA': 'GPS(Guia de Previdência Social)',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 8,
        'TX_CLASS_RADAR': 'Futuro',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    59: {
        'TIPO': 'D',
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
        'CD_CATEGORIA': 59,
        'TX_CATEGORIA': 'Serviços Financeiros',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    60: {
        'TIPO': 'D',
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
        'CD_CATEGORIA': 60,
        'TX_CATEGORIA': 'Serviços Diversos',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    61: {
        'TIPO': 'D',
        'CD_GRUPO': 4,
        'TX_GRUPO': 'Lazer',
        'CD_CATEGORIA': 61,
        'TX_CATEGORIA': 'Jogos e Loterias',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    83: {
        'TIPO': None,
        'CD_GRUPO': 0,
        'TX_GRUPO': 'Sem categoria',
        'CD_CATEGORIA': 83,
        'TX_CATEGORIA': 'Sem Categoria',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 0,
        'TX_CLASS_RADAR': 'Outras Entradas',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    111: {
        'TIPO': 'D',
        'CD_GRUPO': 12,
        'TX_GRUPO': 'Fatura',
        'CD_CATEGORIA': 111,
        'TX_CATEGORIA': 'Cartão de Crédito',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'N'
    },
    279: {
        'TIPO': 'D',
        'CD_GRUPO': 11,
        'TX_GRUPO': 'Outros',
        'CD_CATEGORIA': 279,
        'TX_CATEGORIA': 'Gastos Diversos',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    300: {
        'TIPO': 'C',
        'CD_GRUPO': 14,
        'TX_GRUPO': 'Agro',
        'CD_CATEGORIA': 300,
        'TX_CATEGORIA': 'Receitas Agro',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 1,
        'TX_CLASS_RADAR': 'Renda',
        'IN_AGRO': 'S',
        'IN_PARTICIPA_CALCULO': 'N'
    },
    310: {
        'TIPO': 'D',
        'CD_GRUPO': 14,
        'TX_GRUPO': 'Agro',
        'CD_CATEGORIA': 310,
        'TX_CATEGORIA': 'Criações',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'S',
        'IN_PARTICIPA_CALCULO': 'N'
    },
    330: {
        'TIPO': 'D',
        'CD_GRUPO': 14,
        'TX_GRUPO': 'Agro',
        'CD_CATEGORIA': 330,
        'TX_CATEGORIA': 'Cultivos',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'S',
        'IN_PARTICIPA_CALCULO': 'N'
    },
    350: {
        'TIPO': 'D',
        'CD_GRUPO': 14,
        'TX_GRUPO': 'Agro',
        'CD_CATEGORIA': 350,
        'TX_CATEGORIA': 'Insumos',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'S',
        'IN_PARTICIPA_CALCULO': 'N'
    },
    370: {
        'TIPO': 'D',
        'CD_GRUPO': 14,
        'TX_GRUPO': 'Agro',
        'CD_CATEGORIA': 370,
        'TX_CATEGORIA': 'Apoio Produtivo',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'S',
        'IN_PARTICIPA_CALCULO': 'N'
    },
    3787: {
        'TIPO': 'D',
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
        'CD_CATEGORIA': 3787,
        'TX_CATEGORIA': 'IOF',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    3788: {
        'TIPO': 'D',
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
        'CD_CATEGORIA': 3788,
        'TX_CATEGORIA': 'Encargos e Tarifas',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    3790: {
        'TIPO': 'D',
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
        'CD_CATEGORIA': 3790,
        'TX_CATEGORIA': 'Seguro Residencial',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Não Essenciais',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    4417: {
        'TIPO': 'D',
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
        'CD_CATEGORIA': 4417,
        'TX_CATEGORIA': 'Empréstimos e Prestações',
        'CD_IR': 3,
        'TX_IR': 'Dívidas e ônus reais',
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    39434: {
        'TIPO': 'D',
        'CD_GRUPO': 11,
        'TX_GRUPO': 'Outros',
        'CD_CATEGORIA': 39434,
        'TX_CATEGORIA': 'Cheque',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    39435: {
        'TIPO': 'D',
        'CD_GRUPO': 11,
        'TX_GRUPO': 'Outros',
        'CD_CATEGORIA': 39435,
        'TX_CATEGORIA': 'Saque',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    39436: {
        'TIPO': 'D',
        'CD_GRUPO': 11,
        'TX_GRUPO': 'Outros',
        'CD_CATEGORIA': 39436,
        'TX_CATEGORIA': 'Transferência',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    39437: {
        'TIPO': 'D',
        'CD_GRUPO': 11,
        'TX_GRUPO': 'Outros',
        'CD_CATEGORIA': 39437,
        'TX_CATEGORIA': 'Boletos Diversos',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    448977: {
        'TIPO': 'D',
        'CD_GRUPO': 13,
        'TX_GRUPO': 'Investimentos',
        'CD_CATEGORIA': 448977,
        'TX_CATEGORIA': 'Aplicação',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 8,
        'TX_CLASS_RADAR': 'Futuro',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
    448978: {
        'TIPO': 'C',
        'CD_GRUPO': 13,
        'TX_GRUPO': 'Investimentos',
        'CD_CATEGORIA': 448978,
        'TX_CATEGORIA': 'Resgate de Investimentos',
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
        'CD_CLASS_RADAR': 3,
        'TX_CLASS_RADAR': 'Resgate',
        'IN_AGRO': 'N',
        'IN_PARTICIPA_CALCULO': 'S'
    },
}


### 4.2 Compilador dinâmico de expressões Spark SQL a partir do dicionário

Todo o código SQL do pipeline (classes, grupos, agro, participação, acumuladores e DDL) é gerado dinamicamente a partir de `CATEGORIAS`.


In [ ]:
%%spark

# V13: mapa pequeno de categorias para classificação por broadcast.
# Evita avaliar dezenas de WHENs por transação e elimina as somas de grupos
# que não fazem parte das 72 colunas físicas finais.
_mapa_categoria_radar = [
    (
        str(cat['TIPO']),
        int(codigo),
        int(cat['CD_CLASS_RADAR']),
        1 if cat.get('IN_PARTICIPA_CALCULO') == 'S' else 0,
        1 if cat.get('IN_AGRO') == 'S' else 0,
    )
    for codigo, cat in CATEGORIAS.items()
    if cat.get('TIPO') in ('C', 'D')
       and cat.get('CD_CLASS_RADAR') is not None
]

df_mapa_categoria_radar = spark.createDataFrame(
    _mapa_categoria_radar,
    schema="""
        CD_NTZ_CTB_TRAN STRING,
        CD_CTGR_TRAN_OGNL INT,
        CD_CLASS_RADAR INT,
        IN_PARTICIPA INT,
        IN_AGRO INT
    """,
)
df_mapa_categoria_radar.createOrReplaceTempView("vw_mapa_categoria_radar")

print(
    f"Metadados compilados com sucesso: {len(CATEGORIAS)} categorias; "
    f"{len(_mapa_categoria_radar)} regras C/D disponíveis em vw_mapa_categoria_radar."
)


### 4.3 Datas derivadas da execução

Cálculo de competência, recuo seguro de meses para público e data limite para busca de perfis.


In [ ]:
%%spark

import datetime
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

if hoje:
    data_atual = datetime.date.fromisoformat(str(hoje)[:10])
else:
    data_atual = datetime.date.today()

dt_mes_exea = data_atual.replace(day=1)

def recuar_meses_seguro(dt_referencia, meses):
    ano = dt_referencia.year
    mes = dt_referencia.month - meses
    while mes < 1:
        mes += 12
        ano -= 1
    dia = dt_referencia.day
    if mes in (4, 6, 9, 11) and dia > 30:
        dia = 30
    elif mes == 2:
        bissexto = (ano % 4 == 0 and ano % 100 != 0) or (ano % 400 == 0)
        max_dia = 29 if bissexto else 28
        if dia > max_dia:
            dia = max_dia
    return datetime.date(ano, mes, dia)

data_pbco_ini = recuar_meses_seguro(data_atual, qtd_mes_pbco_alvo)
data_perfil_ini = recuar_meses_seguro(data_atual, 12)

print(f"Data de execução: {data_atual} | Mês de competência (DT_MES_EXEA): {dt_mes_exea}")
print(f"Janela de público ativo ({qtd_mes_pbco_alvo} mês/meses): {data_pbco_ini} a {data_atual}")
print(f"Janela de busca de perfil (12 meses): {data_perfil_ini} a {data_atual}")


## 5. Instrumentação


In [ ]:
%%spark

import time

historico_etapas = []

def registrar_etapa(nome_etapa, momento, inicio=None, linhas=None, particoes=None, detalhes=None):
    agora = time.time()
    if momento == "inicio":
        return agora
    if momento == "fim":
        duracao = round(agora - inicio, 2) if inicio is not None else 0.0
        registro = {
            "etapa": nome_etapa,
            "duracao_segundos": duracao,
            "linhas": linhas,
            "particoes": particoes,
            "detalhes": detalhes or {},
        }
        historico_etapas.append(registro)
        linhas_txt = "-" if linhas is None else str(linhas)
        particoes_txt = "-" if particoes is None else str(particoes)
        print(f"[ETAPA CONCLUÍDA] {nome_etapa:<45} | Duração: {duracao:>7.2f}s | Linhas: {linhas_txt:>10} | Partições: {particoes_txt:>4}")
        return agora


## 6. Público e janela individual no DB2 (Spark SQL)

Leitura fatiada em 4 blocos físicos por `NR_PTC`, registro de views brutas e construção do público e cálculo de ciclo 100% em Spark SQL com join distribuído.


In [ ]:
%%spark

sql_contas_ciclo = """
SELECT
    INTEGER(CD_UOR_CC) AS CD_UOR_CC,
    DECIMAL(NR_CC, 11, 0) AS NR_CC,
    SMALLINT(DD_INC_MM_CLC_BLC) AS DD_INC_MM_CLC_BLC
FROM DB2GFP.CT_GRDR_FNCO
WHERE CD_UOR_CC IS NOT NULL
  AND NR_CC IS NOT NULL
"""

df_contas_ciclo_db2 = conector_db2.sql(
    sql_contas_ciclo,
    fetchsize=10_000,
    query_timeout=15*60,
    partition_column="CD_UOR_CC",
    lower_bound=1,
    upper_bound=10_000,
    num_partitions=16,
)
df_contas_ciclo_db2.createOrReplaceTempView("raw_db2_contas")
print("View bruta de contas (33.8M) registrada com 16 partições JDBC paralelas: raw_db2_contas")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] JDBC CT_GRDR_FNCO — inspeção sem executar leitura adicional
try:
    print("\n[PERF-DIAG] JDBC CT_GRDR_FNCO")
    print("partições Spark/JDBC observadas:", df_contas_ciclo_db2.rdd.getNumPartitions(), "| esperado pela configuração: 16")
    print("schema:", df_contas_ciclo_db2.schema.simpleString())
    print("\n".join([linha for linha in df_contas_ciclo_db2._jdf.queryExecution().executedPlan().toString().splitlines() if any(chave in linha for chave in ("JDBCRelation", "FileScan", "Scan ", "PartitionFilters", "PushedFilters", "Exchange", "Join", "AdaptiveSparkPlan", "InMemoryTableScan"))])[:12000])
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

faixas_ptc_publico = [
    (1, 25, "G1"),
    (26, 50, "G2"),
    (51, 75, "G3"),
    (76, 100, "G4"),
]

dfs_publico_blocos = []
for ptc_min, ptc_max, rotulo in faixas_ptc_publico:
    sql_bloco = f"""
    SELECT
        SMALLINT(NR_PTC) AS NR_PTC,
        INTEGER(CD_CLI) AS CD_CLI,
        DECIMAL(NR_MCA_PCT_OPB, 11, 0) AS NR_MCA_PCT_OPB,
        SMALLINT(CD_PRD) AS CD_PRD,
        SMALLINT(NR_AG_TITR) AS NR_AG_TITR,
        VARCHAR(CD_CT_TITR) AS CD_CT_TITR,
        TIMESTAMP(TS_ATL_TRAN) AS TS_ATL_TRAN,
        DECIMAL(NR_CPF_CNPJ_TITR, 14, 0) AS NR_CPF_ORIGEM
    FROM DB2GFP.TRAN_RLZD_INST_PCT
    WHERE NR_PTC BETWEEN {ptc_min} AND {ptc_max}
      AND CD_EST_TRAN_INST = 0
      AND CD_TIP_PSS = 1
      AND DT_TRAN >= DATE('{data_pbco_ini}')
      AND DT_TRAN <= DATE('{data_atual}')
      AND TS_ATL_TRAN >= TIMESTAMP('{data_pbco_ini} 00:00:00')
      AND TS_ATL_TRAN < TIMESTAMP('{data_atual + datetime.timedelta(days=1)} 00:00:00')
    """
    df_b = (
        conector_db2.sql(
            sql_bloco,
            fetchsize=10_000,
            query_timeout=15*60,
            partition_column="NR_PTC",
            lower_bound=ptc_min,
            upper_bound=ptc_max + 1,
            num_partitions=4,
        )
        .drop("NR_PTC")
    )
    dfs_publico_blocos.append(df_b)

df_publico_bruto = dfs_publico_blocos[0]
for df_b in dfs_publico_blocos[1:]:
    df_publico_bruto = df_publico_bruto.unionByName(df_b)

df_publico_bruto.createOrReplaceTempView("raw_db2_publico_bruto")
print("View bruta do público registrada: 4 blocos físicos x 4 partições JDBC = até 16 fluxos de entrada.")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] JDBC público — somente estrutura/partições; nenhum count
try:
    print("\n[PERF-DIAG] JDBC PUBLICO TRAN_RLZD_INST_PCT")
    print("G1 partições:", dfs_publico_blocos[0].rdd.getNumPartitions(), "| esperado: 4")
    print("G2 partições:", dfs_publico_blocos[1].rdd.getNumPartitions(), "| esperado: 4")
    print("G3 partições:", dfs_publico_blocos[2].rdd.getNumPartitions(), "| esperado: 4")
    print("G4 partições:", dfs_publico_blocos[3].rdd.getNumPartitions(), "| esperado: 4")
    print("UNION bruto partições:", df_publico_bruto.rdd.getNumPartitions(), "| esperado: 16")
    print("Plano de uma leitura JDBC representativa (G1):")
    print("\n".join([linha for linha in dfs_publico_blocos[0]._jdf.queryExecution().executedPlan().toString().splitlines() if any(chave in linha for chave in ("JDBCRelation", "FileScan", "Scan ", "PartitionFilters", "PushedFilters", "Exchange", "Join", "AdaptiveSparkPlan", "InMemoryTableScan"))])[:12000])
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

# A V13 mostrou dois conjuntos de scans da mesma TRAN_RLZD_INST_PCT no plano do público.
# V15 materializa a primeira redução por conta: o DB2 é lido uma vez e os dois ramos reutilizam o cache reduzido.
query_publico_contas_reduzido = """
SELECT
    CD_CLI,
    NR_MCA_PCT_OPB,
    CD_PRD,
    NR_AG_TITR,
    CD_CT_TITR,
    MAX(TS_ATL_TRAN) AS TS_ATL_TRAN_CONTA,
    MIN(NR_CPF_ORIGEM) AS NR_CPF_ORIG_MIN_CONTA,
    MAX(NR_CPF_ORIGEM) AS NR_CPF_ORIG_MAX_CONTA,
    MIN(CASE WHEN NR_CPF_ORIGEM BETWEEN 1 AND 99999999999 THEN NR_CPF_ORIGEM END) AS NR_CPF_MIN_CONTA,
    MAX(CASE WHEN NR_CPF_ORIGEM BETWEEN 1 AND 99999999999 THEN NR_CPF_ORIGEM END) AS NR_CPF_MAX_CONTA,
    CASE
        WHEN NR_MCA_PCT_OPB = 999999999
         AND CD_PRD = 6
         AND NR_AG_TITR IS NOT NULL
         AND CD_CT_TITR IS NOT NULL
         AND TRIM(CD_CT_TITR) != ''
        THEN 1 ELSE 0
    END AS IN_CONTA_ELEGIVEL
FROM raw_db2_publico_bruto
GROUP BY CD_CLI, NR_MCA_PCT_OPB, CD_PRD, NR_AG_TITR, CD_CT_TITR
"""

df_publico_contas_reduzido = spark.sql(query_publico_contas_reduzido).persist(StorageLevel.MEMORY_AND_DISK)
df_publico_contas_reduzido.createOrReplaceTempView("vw_publico_contas_reduzido")

id_reducao_publico = "PUBLICO_REDUCAO_CONTAS_DB2"
inicio_reducao_publico = registrar_etapa(id_reducao_publico, "inicio")
qt_publico_contas_reduzido = df_publico_contas_reduzido.count()
registrar_etapa(
    id_reducao_publico,
    "fim",
    inicio=inicio_reducao_publico,
    linhas=qt_publico_contas_reduzido,
    particoes=df_publico_contas_reduzido.rdd.getNumPartitions(),
)
print(f"Redução comum do público materializada: {qt_publico_contas_reduzido:,} contas/chaves reduzidas; scans JDBC não serão repetidos nos ramos seguintes.")

query_publico_janela = f"""
WITH publico_cliente AS (
    SELECT
        CD_CLI,
        MAX(TS_ATL_TRAN_CONTA) AS TS_ATL_TRAN,
        MIN(NR_CPF_ORIG_MIN_CONTA) AS NR_CPF_ORIG_MIN,
        MAX(NR_CPF_ORIG_MAX_CONTA) AS NR_CPF_ORIG_MAX,
        MIN(NR_CPF_MIN_CONTA) AS NR_CPF_MIN,
        MAX(NR_CPF_MAX_CONTA) AS NR_CPF_MAX
    FROM vw_publico_contas_reduzido
    GROUP BY CD_CLI
),
contas_elegiveis_base AS (
    SELECT
        CD_CLI,
        CAST(NR_AG_TITR AS INT) AS CD_UOR_CC,
        TRIM(CD_CT_TITR) AS CD_CT_TITR_TRIM
    FROM vw_publico_contas_reduzido
    WHERE IN_CONTA_ELEGIVEL = 1
),
contas_elegiveis AS (
    SELECT
        CD_CLI,
        CD_UOR_CC,
        CASE
            WHEN CD_CT_TITR_TRIM RLIKE '^[0-9]+$'
             AND LENGTH(REGEXP_REPLACE(CD_CT_TITR_TRIM, '^0+', '')) <= 11
            THEN CAST(REGEXP_REPLACE(CD_CT_TITR_TRIM, '^0+', '') AS DECIMAL(11,0))
            ELSE NULL
        END AS NR_CC
    FROM contas_elegiveis_base
),
ciclo_cliente AS (
    SELECT
        e.CD_CLI,
        COUNT(1) AS QT_CONTA_ELEGIVEL,
        MAX(c.DD_INC_MM_CLC_BLC) AS DD_INC_MM_CLC_BLC_CONTA
    FROM contas_elegiveis e
    LEFT JOIN raw_db2_contas c
      ON e.CD_UOR_CC = c.CD_UOR_CC
     AND e.NR_CC = c.NR_CC
    GROUP BY e.CD_CLI
),
publico_atributos AS (
    SELECT
        p.CD_CLI,
        p.TS_ATL_TRAN,
        CAST(1 AS SMALLINT) AS CD_TIP_PSS,
        CAST(1 AS INT) AS QT_TIP_PSS_DIST,
        CASE
            WHEN p.NR_CPF_ORIG_MAX IS NULL THEN 0
            WHEN p.NR_CPF_ORIG_MIN = p.NR_CPF_ORIG_MAX THEN 1
            ELSE 2
        END AS QT_CPF_ORIG_DIST,
        CASE
            WHEN p.NR_CPF_MAX IS NULL THEN 0
            WHEN p.NR_CPF_MIN = p.NR_CPF_MAX THEN 1
            ELSE 2
        END AS QT_CPF_VALIDO_DIST,
        CAST(p.NR_CPF_MAX AS DECIMAL(11,0)) AS NR_CPF,
        CAST(
            CASE
                WHEN COALESCE(c.QT_CONTA_ELEGIVEL, 0) = 0 THEN 997
                WHEN c.QT_CONTA_ELEGIVEL > 1 THEN 996
                WHEN c.DD_INC_MM_CLC_BLC_CONTA IS NULL THEN 999
                ELSE c.DD_INC_MM_CLC_BLC_CONTA
            END AS SMALLINT
        ) AS DD_INC_MM_CLC_BLC,
        CAST(
            CASE
                WHEN c.QT_CONTA_ELEGIVEL = 1
                 AND c.DD_INC_MM_CLC_BLC_CONTA BETWEEN 1 AND 31
                THEN c.DD_INC_MM_CLC_BLC_CONTA
                ELSE 1
            END AS INT
        ) AS DD_CICLO_CALCULO
    FROM publico_cliente p
    LEFT JOIN ciclo_cliente c ON p.CD_CLI = c.CD_CLI
),
publico_datas AS (
    SELECT
        a.*,
        TRUNC(TO_DATE(TS_ATL_TRAN), 'month') AS DT_MES_REFERENCIA,
        DATE_ADD(
            TRUNC(TO_DATE(TS_ATL_TRAN), 'month'),
            CAST(LEAST(DD_CICLO_CALCULO, DAYOFMONTH(LAST_DAY(TRUNC(TO_DATE(TS_ATL_TRAN), 'month')))) - 1 AS INT)
        ) AS DT_INI_CICLO_MES_REFERENCIA
    FROM publico_atributos a
),
publico_ciclo_aberto AS (
    SELECT
        d.*,
        CASE
            WHEN TO_DATE(TS_ATL_TRAN) >= DT_INI_CICLO_MES_REFERENCIA
            THEN DT_INI_CICLO_MES_REFERENCIA
            ELSE DATE_ADD(
                ADD_MONTHS(DT_MES_REFERENCIA, -1),
                CAST(LEAST(DD_CICLO_CALCULO, DAYOFMONTH(LAST_DAY(ADD_MONTHS(DT_MES_REFERENCIA, -1)))) - 1 AS INT)
            )
        END AS DT_INI_CICLO_ABERTO
    FROM publico_datas d
),
publico_janela_final AS (
    SELECT
        c.CD_CLI,
        c.TS_ATL_TRAN,
        c.CD_TIP_PSS,
        c.QT_TIP_PSS_DIST,
        c.QT_CPF_ORIG_DIST,
        c.QT_CPF_VALIDO_DIST,
        c.NR_CPF,
        CAST(
            CASE
                WHEN c.QT_CPF_ORIG_DIST > 1 THEN -4
                WHEN c.QT_CPF_ORIG_DIST = 0 THEN -2
                WHEN c.QT_CPF_VALIDO_DIST = 0 THEN -2
                ELSE 0
            END AS SMALLINT
        ) AS CD_STS_CPF,
        c.DD_INC_MM_CLC_BLC,
        TO_DATE(DATE_ADD(
            ADD_MONTHS(TRUNC(c.DT_INI_CICLO_ABERTO, 'month'), -{periodo}),
            CAST(LEAST(c.DD_CICLO_CALCULO, DAYOFMONTH(LAST_DAY(ADD_MONTHS(TRUNC(c.DT_INI_CICLO_ABERTO, 'month'), -{periodo})))) - 1 AS INT)
        )) AS DT_REF_INI,
        TO_DATE(DATE_SUB(c.DT_INI_CICLO_ABERTO, 1)) AS DT_REF_FIM
    FROM publico_ciclo_aberto c
)
SELECT
    CAST(CD_CLI AS INT) AS CD_CLI,
    TS_ATL_TRAN,
    CD_TIP_PSS,
    QT_TIP_PSS_DIST,
    QT_CPF_ORIG_DIST,
    QT_CPF_VALIDO_DIST,
    NR_CPF,
    CD_STS_CPF,
    DD_INC_MM_CLC_BLC,
    DT_REF_INI,
    DT_REF_FIM
FROM publico_janela_final
"""

df_publico_janela = spark.sql(query_publico_janela).persist(StorageLevel.MEMORY_AND_DISK)
df_publico_janela.createOrReplaceTempView("vw_janela_financeira")
print("Público e janela preparados sobre a redução materializada; a fonte transacional JDBC não será relida pelos dois ramos.")


In [ ]:
%%spark

id_etapa_pbco = "PUBLICO_JANELA_DB2"
inicio_pbco = registrar_etapa(id_etapa_pbco, "inicio")
try:
    qt_publico_janela = df_publico_janela.count()
except Exception as exc:
    print(f"Erro na leitura do público DB2: {exc}")
    raise
registrar_etapa(
    id_etapa_pbco,
    "fim",
    inicio=inicio_pbco,
    linhas=qt_publico_janela,
    particoes=df_publico_janela.rdd.getNumPartitions(),
)
if qt_publico_janela == 0:
    raise ValueError(f"O público ativo de {qtd_mes_pbco_alvo} mês/meses está vazio.")

df_publico_contas_reduzido.unpersist()
print(f"Público ativo materializado com sucesso: {qt_publico_janela:,} clientes. Redução intermediária liberada.")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Público materializado — metadados locais, sem ação adicional
try:
    print("\n[PERF-DIAG] PUBLICO/JANELA MATERIALIZADO")
    print("partições finais:", df_publico_janela.rdd.getNumPartitions())
    print("storageLevel:", df_publico_janela.storageLevel)
    print("view cacheada:", spark.catalog.isCached("vw_janela_financeira"))
    print("redução comum do público liberada:", not bool(df_publico_contas_reduzido.storageLevel.useMemory or df_publico_contas_reduzido.storageLevel.useDisk))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


## 7. Perfil financeiro no DB2 (Estratégia Fail-Fast em Spark SQL)


In [ ]:
%%spark

sql_perfil_financeiro_db2 = f"""
SELECT
    d.DT_REF AS DT_REF,
    d.CD_CLI AS CD_CLI,
    d.CD_MAC_PRFL_CLI AS CD_MAC_PRFL_CLI,
    d.CD_MIC_PRFL_CLI AS CD_MIC_PRFL_CLI
FROM DB2D1D.DVS_GRDR_FNCO_PF d
WHERE d.DT_REF >= DATE('{data_perfil_ini}')
  AND d.DT_REF < DATE('{data_atual + datetime.timedelta(days=1)}')
  AND d.CD_CLI BETWEEN 1 AND 999999999
"""
df_perfil_financeiro_db2_bruto = conector_db2.sql(
    sql_perfil_financeiro_db2,
    fetchsize=10_000,
    query_timeout=15 * 60,
    partition_column="DT_REF",
    lower_bound=data_perfil_ini,
    upper_bound=data_atual + datetime.timedelta(days=1),
    num_partitions=16,
)
df_perfil_financeiro_db2_bruto.createOrReplaceTempView("raw_db2_perfil_bruto")
print("View bruta de perfil registrada com 16 partições JDBC por DT_REF: raw_db2_perfil_bruto")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] JDBC perfil financeiro — inspeção sem count adicional
try:
    print("\n[PERF-DIAG] JDBC DVS_GRDR_FNCO_PF")
    print("partições Spark/JDBC observadas:", df_perfil_financeiro_db2_bruto.rdd.getNumPartitions(), "| esperado: 16")
    print("schema:", df_perfil_financeiro_db2_bruto.schema.simpleString())
    print("\n".join([linha for linha in df_perfil_financeiro_db2_bruto._jdf.queryExecution().executedPlan().toString().splitlines() if any(chave in linha for chave in ("JDBCRelation", "FileScan", "Scan ", "PartitionFilters", "PushedFilters", "Exchange", "Join", "AdaptiveSparkPlan", "InMemoryTableScan"))])[:12000])
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

query_perfil_financeiro = """
WITH perfil_ordenado AS (
    SELECT
        p.CD_CLI,
        p.CD_MAC_PRFL_CLI,
        p.CD_MIC_PRFL_CLI,
        ROW_NUMBER() OVER (
            PARTITION BY p.CD_CLI 
            ORDER BY p.DT_REF DESC, p.CD_MAC_PRFL_CLI DESC, p.CD_MIC_PRFL_CLI DESC
        ) AS _NR_ORDEM
    FROM raw_db2_perfil_bruto p
    INNER JOIN (SELECT DISTINCT CD_CLI FROM vw_janela_financeira) j
        ON p.CD_CLI = j.CD_CLI
),
perfil_dedup AS (
    SELECT CD_CLI, CD_MAC_PRFL_CLI, CD_MIC_PRFL_CLI
    FROM perfil_ordenado
    WHERE _NR_ORDEM = 1
),
perfil_nomeado AS (
    SELECT
        CAST(CD_CLI AS BIGINT) AS CD_CLI,
        CAST(CD_MAC_PRFL_CLI AS BIGINT) AS CD_MAC_PRFL_CLI,
        CASE
            WHEN CD_MAC_PRFL_CLI IS NULL THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 'Endividado'
            WHEN CD_MAC_PRFL_CLI = 2 THEN 'Equilibrista'
            WHEN CD_MAC_PRFL_CLI = 3 THEN 'Investidor'
            ELSE 'CODIGO NAO MAPEADO'
        END AS NM_MAC_PRFL_CLI,
        CAST(CD_MIC_PRFL_CLI AS BIGINT) AS CD_MIC_PRFL_CLI,
        CASE
            WHEN CD_MIC_PRFL_CLI IS NULL THEN NULL
            WHEN CD_MIC_PRFL_CLI = 1 THEN 'Inadimplente'
            WHEN CD_MIC_PRFL_CLI = 2 THEN 'Acrobata'
            WHEN CD_MIC_PRFL_CLI = 3 THEN 'Iminente'
            WHEN CD_MIC_PRFL_CLI = 4 THEN 'Consciente'
            WHEN CD_MIC_PRFL_CLI = 5 THEN 'Equilibrista'
            WHEN CD_MIC_PRFL_CLI = 6 THEN 'Acelerado'
            WHEN CD_MIC_PRFL_CLI = 7 THEN 'Precavido'
            WHEN CD_MIC_PRFL_CLI = 8 THEN 'Despreocupado'
            ELSE 'CODIGO NAO MAPEADO'
        END AS NM_MIC_PRFL_CLI
    FROM perfil_dedup
)
SELECT
    CD_CLI,
    CD_MAC_PRFL_CLI,
    NM_MAC_PRFL_CLI,
    CD_MIC_PRFL_CLI,
    NM_MIC_PRFL_CLI,
    CASE
        WHEN NM_MAC_PRFL_CLI IS NULL OR NM_MIC_PRFL_CLI IS NULL THEN CAST(NULL AS STRING)
        WHEN NM_MAC_PRFL_CLI = 'CODIGO NAO MAPEADO' OR NM_MIC_PRFL_CLI = 'CODIGO NAO MAPEADO' THEN 'CODIGO NAO MAPEADO'
        WHEN NM_MAC_PRFL_CLI = 'Equilibrista' AND NM_MIC_PRFL_CLI = 'Equilibrista' THEN 'Equilibrista'
        ELSE CONCAT(NM_MAC_PRFL_CLI, ' ', NM_MIC_PRFL_CLI)
    END AS NM_PRFL_FIN
FROM perfil_nomeado
"""

df_perfil_financeiro = spark.sql(query_perfil_financeiro).persist(StorageLevel.MEMORY_AND_DISK)
df_perfil_financeiro.createOrReplaceTempView("vw_perfil_financeiro")

id_perfil_db2 = "PERFIL_FINANCEIRO_DB2"
inicio_perfil_db2 = registrar_etapa(id_perfil_db2, "inicio")
try:
    qt_perfil_db2 = df_perfil_financeiro.count()
except Exception as exc:
    df_perfil_financeiro.unpersist()
    print(f"Erro no perfil DB2: {exc}")
    raise
registrar_etapa(
    id_perfil_db2,
    "fim",
    inicio=inicio_perfil_db2,
    linhas=qt_perfil_db2,
    particoes=df_perfil_financeiro.rdd.getNumPartitions(),
)
print(f"Perfil financeiro materializado via Spark SQL: {qt_perfil_db2:,} registros mais recentes vinculados ao público (vw_perfil_financeiro).")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] PERFIL FINANCEIRO MATERIALIZADO — metadados locais
try:
    print("\n[PERF-DIAG] PERFIL FINANCEIRO MATERIALIZADO")
    print("partições:", df_perfil_financeiro.rdd.getNumPartitions())
    print("storageLevel:", df_perfil_financeiro.storageLevel)
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


## 8. Renda presumida no Hive (Estratégia Fail-Fast em Spark SQL)


In [ ]:
%%spark

id_particoes_rdpr = "METADADOS_PARTICOES_RDPR"
inicio_particoes_rdpr = registrar_etapa(id_particoes_rdpr, "inicio")

particoes_rdpr = {}
try:
    df_particoes_max = spark.sql(f"""
        SELECT
            TX_CTGR_MOD AS TX_CTGR_MOD,
            MAX(DT_REF) AS MAX_DT_REF
        FROM HIVE_D1Q.RDPR_PF
        WHERE DT_REF <= DATE('{data_atual}')
          AND TX_CTGR_MOD IN ('behaviour', 'bureau')
        GROUP BY TX_CTGR_MOD
    """).collect()

    for row in df_particoes_max:
        if row["TX_CTGR_MOD"] and row["MAX_DT_REF"]:
            particoes_rdpr[row["TX_CTGR_MOD"]] = row["MAX_DT_REF"]
except Exception as exc:
    print(f"Erro ao consultar partições RDPR em HIVE_D1Q: {exc}")
    raise

registrar_etapa(id_particoes_rdpr, "fim", inicio=inicio_particoes_rdpr, detalhes=particoes_rdpr)

categorias_sem_particao = [c for c in ("behaviour", "bureau") if c not in particoes_rdpr]
if categorias_sem_particao:
    raise ValueError(f"RDPR_PF sem partição válida até {data_atual}: {categorias_sem_particao}")

dt_ref_behaviour = particoes_rdpr["behaviour"]
dt_ref_bureau = particoes_rdpr["bureau"]
print(f"Partições RDPR selecionadas: behaviour={dt_ref_behaviour} | bureau={dt_ref_bureau}")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] RDPR_PF — plano resumido compatível; não executa nova leitura
try:
    print("\n[PERF-DIAG] HIVE RDPR_PF — PLANO DE SELEÇÃO DAS PARTIÇÕES")
    print("\n".join([linha for linha in spark.sql(f"""
        SELECT TX_CTGR_MOD, MAX(DT_REF) AS MAX_DT_REF
        FROM HIVE_D1Q.RDPR_PF
        WHERE DT_REF <= DATE('{data_atual}')
          AND TX_CTGR_MOD IN ('behaviour', 'bureau')
        GROUP BY TX_CTGR_MOD
    """)._jdf.queryExecution().executedPlan().toString().splitlines() if any(chave in linha for chave in ("FileScan", "Scan ", "PartitionFilters", "PushedFilters", "Exchange", "AdaptiveSparkPlan"))])[:12000])
except Exception as exc:
    print(f"[PERF-DIAG] AVISO PLANO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

df_cpfs_publico_validos = spark.sql("""
    SELECT DISTINCT NR_CPF
    FROM vw_janela_financeira
    WHERE CD_STS_CPF = 0
""").persist(StorageLevel.MEMORY_AND_DISK)
df_cpfs_publico_validos.createOrReplaceTempView("vw_cpfs_publico_validos")

id_cpfs_validos = "CONTA_CPFS_VALIDOS_PUBLICO"
inicio_cpfs_validos = registrar_etapa(id_cpfs_validos, "inicio")
qt_cpfs_publico_validos = df_cpfs_publico_validos.count()
registrar_etapa(id_cpfs_validos, "fim", inicio=inicio_cpfs_validos, linhas=qt_cpfs_publico_validos)
print(f"CPFs distintos e válidos do público: {qt_cpfs_publico_validos:,} (vw_cpfs_publico_validos).")


In [ ]:
%%spark

query_rdpr_behaviour = f"""
WITH behaviour_filtrado AS (
    SELECT
        CAST(r.NR_CPF AS DECIMAL(11,0)) AS NR_CPF,
        CAST(r.VL_RDPR AS DECIMAL(15,2)) AS VL_RDPR
    FROM HIVE_D1Q.RDPR_PF r
    INNER JOIN vw_cpfs_publico_validos cp
        ON CAST(r.NR_CPF AS DECIMAL(11,0)) = cp.NR_CPF
    WHERE r.DT_REF = DATE('{dt_ref_behaviour}')
      AND r.TX_CTGR_MOD = 'behaviour'
      AND r.CD_IN_UTZO = -1
),
behaviour_agrupado AS (
    SELECT
        NR_CPF,
        COUNT(1) AS QT_REGISTROS,
        MAX(VL_RDPR) AS VL_RDPR_ORIGEM
    FROM behaviour_filtrado
    GROUP BY NR_CPF
)
SELECT
    NR_CPF,
    CAST(
        CASE
            WHEN QT_REGISTROS > 1 THEN -5
            WHEN VL_RDPR_ORIGEM IS NULL THEN -6
            WHEN VL_RDPR_ORIGEM < 0 THEN -7
            ELSE 0
        END AS SMALLINT
    ) AS CD_STS_MODELO,
    VL_RDPR_ORIGEM
FROM behaviour_agrupado
"""

df_rdpr_behaviour_avaliado = spark.sql(query_rdpr_behaviour).persist(StorageLevel.MEMORY_AND_DISK)
df_rdpr_behaviour_avaliado.createOrReplaceTempView("vw_rdpr_behaviour")

id_avalia_behaviour = "AVALIA_RDPR_BEHAVIOUR"
inicio_avalia_behaviour = registrar_etapa(id_avalia_behaviour, "inicio")
qt_rdpr_behaviour_avaliado = df_rdpr_behaviour_avaliado.count()
registrar_etapa(id_avalia_behaviour, "fim", inicio=inicio_avalia_behaviour, linhas=qt_rdpr_behaviour_avaliado)
print(f"Behaviour avaliado via Spark SQL: {qt_rdpr_behaviour_avaliado:,} registros (vw_rdpr_behaviour).")


In [ ]:
%%spark

query_rdpr_bureau = f"""
WITH bureau_filtrado AS (
    SELECT
        CAST(r.NR_CPF AS DECIMAL(11,0)) AS NR_CPF,
        CAST(r.VL_RDPR AS DECIMAL(15,2)) AS VL_RDPR
    FROM HIVE_D1Q.RDPR_PF r
    INNER JOIN vw_cpfs_publico_validos cp
        ON CAST(r.NR_CPF AS DECIMAL(11,0)) = cp.NR_CPF
    WHERE r.DT_REF = DATE('{dt_ref_bureau}')
      AND r.TX_CTGR_MOD = 'bureau'
      AND r.CD_IN_UTZO = -1
),
bureau_agrupado AS (
    SELECT
        NR_CPF,
        COUNT(1) AS QT_REGISTROS,
        MAX(VL_RDPR) AS VL_RDPR_ORIGEM
    FROM bureau_filtrado
    GROUP BY NR_CPF
)
SELECT
    NR_CPF,
    CAST(
        CASE
            WHEN QT_REGISTROS > 1 THEN -5
            WHEN VL_RDPR_ORIGEM IS NULL THEN -6
            WHEN VL_RDPR_ORIGEM < 0 THEN -7
            ELSE 0
        END AS SMALLINT
    ) AS CD_STS_MODELO,
    VL_RDPR_ORIGEM
FROM bureau_agrupado
"""

df_rdpr_bureau_avaliado = spark.sql(query_rdpr_bureau).persist(StorageLevel.MEMORY_AND_DISK)
df_rdpr_bureau_avaliado.createOrReplaceTempView("vw_rdpr_bureau")

id_avalia_bureau = "AVALIA_RDPR_BUREAU"
inicio_avalia_bureau = registrar_etapa(id_avalia_bureau, "inicio")
qt_rdpr_bureau_avaliado = df_rdpr_bureau_avaliado.count()
registrar_etapa(id_avalia_bureau, "fim", inicio=inicio_avalia_bureau, linhas=qt_rdpr_bureau_avaliado)
print(f"Bureau avaliado via Spark SQL: {qt_rdpr_bureau_avaliado:,} registros (vw_rdpr_bureau).")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] RDPR Behaviour/Bureau — metadados locais
try:
    print("\n[PERF-DIAG] RDPR BEHAVIOUR/BUREAU")
    print("behaviour partições:", df_rdpr_behaviour_avaliado.rdd.getNumPartitions(), "| storage:", df_rdpr_behaviour_avaliado.storageLevel)
    print("bureau partições:", df_rdpr_bureau_avaliado.rdd.getNumPartitions(), "| storage:", df_rdpr_bureau_avaliado.storageLevel)
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

query_renda_presumida = """
WITH modelo_escolhido AS (
    SELECT
        cp.NR_CPF,
        CAST(
            CASE
                WHEN bh.NR_CPF IS NOT NULL THEN bh.CD_STS_MODELO
                WHEN bu.NR_CPF IS NOT NULL THEN bu.CD_STS_MODELO
                ELSE -1
            END AS SMALLINT
        ) AS CD_STS_MODELO,
        CASE
            WHEN bh.NR_CPF IS NOT NULL THEN bh.VL_RDPR_ORIGEM
            WHEN bu.NR_CPF IS NOT NULL THEN bu.VL_RDPR_ORIGEM
            ELSE NULL
        END AS VL_RDPR_ORIGEM,
        CASE
            WHEN bh.NR_CPF IS NOT NULL THEN 'behaviour'
            WHEN bu.NR_CPF IS NOT NULL THEN 'bureau'
            ELSE NULL
        END AS TX_MODELO_RENDA
    FROM vw_cpfs_publico_validos cp
    LEFT JOIN vw_rdpr_behaviour bh ON cp.NR_CPF = bh.NR_CPF
    LEFT JOIN vw_rdpr_bureau bu ON cp.NR_CPF = bu.NR_CPF
),
modelo_com_valor AS (
    SELECT
        NR_CPF,
        CD_STS_MODELO,
        TX_MODELO_RENDA,
        CAST(
            CASE
                WHEN CD_STS_MODELO = 0 THEN CAST(VL_RDPR_ORIGEM AS DECIMAL(18,2))
                ELSE CAST(CD_STS_MODELO AS DECIMAL(18,2))
            END AS DECIMAL(18,2)
        ) AS VL_REN_PRES_MODELO
    FROM modelo_escolhido
)
SELECT
    j.CD_CLI,
    j.NR_CPF,
    j.CD_STS_CPF,
    CAST(
        CASE
            WHEN j.CD_STS_CPF < 0 THEN CAST(j.CD_STS_CPF AS DECIMAL(18,2))
            WHEN m.VL_REN_PRES_MODELO IS NOT NULL THEN m.VL_REN_PRES_MODELO
            ELSE CAST(-1 AS DECIMAL(18,2))
        END AS DECIMAL(18,2)
    ) AS VL_REN_PRES,
    m.TX_MODELO_RENDA
FROM vw_janela_financeira j
LEFT JOIN modelo_com_valor m ON j.NR_CPF = m.NR_CPF
"""

df_renda_presumida = spark.sql(query_renda_presumida).persist(StorageLevel.MEMORY_AND_DISK)
df_renda_presumida.createOrReplaceTempView("vw_renda_presumida")

id_renda_cliente = "MATERIALIZA_RENDA_PRESUMIDA_CLIENTE"
inicio_renda_cliente = registrar_etapa(id_renda_cliente, "inicio")
qt_renda_cliente = df_renda_presumida.count()
registrar_etapa(id_renda_cliente, "fim", inicio=inicio_renda_cliente, linhas=qt_renda_cliente)

df_rdpr_behaviour_avaliado.unpersist()
df_rdpr_bureau_avaliado.unpersist()
df_cpfs_publico_validos.unpersist()

print(f"Renda presumida final vinculada via Spark SQL: {qt_renda_cliente:,} clientes (vw_renda_presumida).")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] RENDA PRESUMIDA — metadados locais
try:
    print("\n[PERF-DIAG] RENDA PRESUMIDA")
    print("partições:", df_renda_presumida.rdd.getNumPartitions())
    print("storageLevel:", df_renda_presumida.storageLevel)
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


## 9. Envelope mensal e transações da janela — V15

A execução anterior mostrou que o acumulador recursivo consumia a maior parte do tempo. Na V15:

1. o envelope é cortado pelos limites reais da janela;
2. cada mês ativo usa no máximo 16 fluxos JDBC, é agregado e materializado uma única vez;
3. somente depois de todos os meses, os agregados mensais já cacheados são unidos e consolidados por `CD_CLI` **uma única vez**.


In [ ]:
%%spark

limites_janela = spark.sql("""
    SELECT
        MIN(DT_REF_INI) AS DT_REF_INI_GLOBAL,
        MAX(DT_REF_FIM) AS DT_REF_FIM_GLOBAL
    FROM vw_janela_financeira
""").first()

dt_ref_ini_global = limites_janela["DT_REF_INI_GLOBAL"]
dt_ref_fim_global = limites_janela["DT_REF_FIM_GLOBAL"]

if dt_ref_ini_global is None or dt_ref_fim_global is None:
    raise ValueError("Não foi possível determinar os limites globais da janela.")
if dt_ref_ini_global > dt_ref_fim_global:
    raise ValueError(f"Envelope inválido: início={dt_ref_ini_global} > fim={dt_ref_fim_global}")

cursor_mes = dt_ref_ini_global.replace(day=1)
meses_envelope = []
while cursor_mes <= dt_ref_fim_global and len(meses_envelope) < 9:
    prox_cursor = (cursor_mes.replace(day=28) + datetime.timedelta(days=4)).replace(day=1)
    ultimo_dia_mes = prox_cursor - datetime.timedelta(days=1)
    inicio_recorte = max(cursor_mes, dt_ref_ini_global)
    fim_recorte = min(ultimo_dia_mes, dt_ref_fim_global)
    meses_envelope.append((inicio_recorte.isoformat(), fim_recorte.isoformat()))
    cursor_mes = prox_cursor

if cursor_mes <= dt_ref_fim_global:
    raise ValueError("O envelope ultrapassou o limite operacional de 9 meses.")

dfs_mensais_materializados = []
print(f"Envelope global exato: {dt_ref_ini_global} a {dt_ref_fim_global} ({len(meses_envelope)} meses identificados).")
print("Recortes JDBC mensais:", meses_envelope)


### 9.1. Fragmento mensal 01

Leitura DB2 limitada ao recorte real do envelope, 4 blocos × 4 partições JDBC. O agregado mensal é materializado uma única vez e fica disponível para a consolidação final.


In [ ]:
%%spark

fragmento_01_ativo = len(meses_envelope) >= 1
df_mes_01 = None

if fragmento_01_ativo:
    mes_inicio_01, mes_fim_01 = meses_envelope[0]
    faixas_ptc_mes_01 = [
        (1, 25, "G1"),
        (26, 50, "G2"),
        (51, 75, "G3"),
        (76, 100, "G4"),
    ]
    dfs_db2_01 = []
    for ptc_min, ptc_max, rotulo in faixas_ptc_mes_01:
        sql_grupo = f"""
        SELECT
            SMALLINT(NR_PTC) AS NR_PTC,
            INTEGER(CD_CLI) AS CD_CLI,
            DT_TRAN AS DT_TRAN,
            DECIMAL(VL_TRAN, 15, 2) AS VL_TRAN,
            CHAR(CD_NTZ_CTB_TRAN) AS CD_NTZ_CTB_TRAN,
            INTEGER(CD_CTGR_TRAN_OGNL) AS CD_CTGR_TRAN_OGNL,
            CHAR(CD_TIP_MOE_CRR) AS CD_TIP_MOE_CRR
        FROM DB2GFP.TRAN_RLZD_INST_PCT
        WHERE NR_PTC BETWEEN {ptc_min} AND {ptc_max}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND DT_TRAN >= DATE('{mes_inicio_01}')
          AND DT_TRAN <= DATE('{mes_fim_01}')
        """
        df_g = (
            conector_db2.sql(
                sql_grupo,
                fetchsize=10_000,
                query_timeout=15*60,
                partition_column="NR_PTC",
                lower_bound=ptc_min,
                upper_bound=ptc_max + 1,
                num_partitions=4,
            )
            .drop("NR_PTC")
        )
        dfs_db2_01.append((rotulo, df_g))
    print(f"Mês 01 ({mes_inicio_01} a {mes_fim_01}): 4 blocos x 4 partições JDBC preparados.")
else:
    print("Mês 01 inativo no envelope atual.")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Mês 01 — partições JDBC, sem ação de dados
try:
    if fragmento_01_ativo:
        print("\n[PERF-DIAG] JDBC MÊS 01")
        print("G1:", dfs_db2_01[0][1].rdd.getNumPartitions(), "| G2:", dfs_db2_01[1][1].rdd.getNumPartitions(), "| G3:", dfs_db2_01[2][1].rdd.getNumPartitions(), "| G4:", dfs_db2_01[3][1].rdd.getNumPartitions())
        print("total potencial de fluxos JDBC simultâneos:", sum(item[1].rdd.getNumPartitions() for item in dfs_db2_01))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

if fragmento_01_ativo:
    df_mes_01_bruto = dfs_db2_01[0][1]
    for _, df_grupo in dfs_db2_01[1:]:
        df_mes_01_bruto = df_mes_01_bruto.unionByName(df_grupo)

    df_mes_01_bruto.createOrReplaceTempView("raw_db2_trans_mes_01")

    query_mes_01 = """WITH transacoes_filtradas AS (
    SELECT /*+ BROADCAST(m) */
        t.CD_CLI,
        t.DT_TRAN,
        t.VL_TRAN,
        t.CD_NTZ_CTB_TRAN,
        t.CD_CTGR_TRAN_OGNL,
        t.CD_TIP_MOE_CRR,
        COALESCE(m.CD_CLASS_RADAR, CASE WHEN t.CD_NTZ_CTB_TRAN = 'C' THEN 0 WHEN t.CD_NTZ_CTB_TRAN = 'D' THEN 5 END) AS _CD_CLASS_RADAR,
        COALESCE(m.IN_PARTICIPA, 0) AS _IN_PARTICIPA,
        COALESCE(m.IN_AGRO, 0) AS _IN_AGRO
    FROM raw_db2_trans_mes_01 t
    INNER JOIN vw_janela_financeira j
        ON t.CD_CLI = j.CD_CLI
    LEFT JOIN vw_mapa_categoria_radar m
        ON t.CD_NTZ_CTB_TRAN = m.CD_NTZ_CTB_TRAN
       AND t.CD_CTGR_TRAN_OGNL = m.CD_CTGR_TRAN_OGNL
    WHERE t.DT_TRAN >= j.DT_REF_INI
      AND t.DT_TRAN <= j.DT_REF_FIM
)
SELECT
    CD_CLI,
    COUNT(1) AS QT_TRANS_TOTAL,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_TRANS_ENT,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_TRANS_SAI,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 1 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 2 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 3 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 0 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 4 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_CRED,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 5 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 6 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 7 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 8 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 9 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_OBR,
    CAST(MAX(_IN_AGRO) AS INT) AS IN_AGRO,
    CAST(SUM(CASE WHEN COALESCE(TRIM(CD_TIP_MOE_CRR), '') != 'BRL' THEN 1 ELSE 0 END) AS BIGINT) AS QT_MOE_NAO_BRL
FROM transacoes_filtradas
GROUP BY CD_CLI"""
    df_mes_01 = spark.sql(query_mes_01).persist(StorageLevel.MEMORY_AND_DISK)
    df_mes_01.createOrReplaceTempView("vw_mes_01")

    id_etapa_01 = f"TRANSACOES_MES_01_{mes_inicio_01[:7]}"
    inicio_mes_01 = registrar_etapa(id_etapa_01, "inicio")
    qt_mes_01 = df_mes_01.count()
    registrar_etapa(
        id_etapa_01,
        "fim",
        inicio=inicio_mes_01,
        linhas=qt_mes_01,
        particoes=df_mes_01.rdd.getNumPartitions(),
    )
    dfs_mensais_materializados.append(df_mes_01)
    print(f"Mês 01 materializado uma única vez: {qt_mes_01:,} clientes no agregado mensal.")


### 9.2. Fragmento mensal 02

Leitura DB2 limitada ao recorte real do envelope, 4 blocos × 4 partições JDBC. O agregado mensal é materializado uma única vez e fica disponível para a consolidação final.


In [ ]:
%%spark

fragmento_02_ativo = len(meses_envelope) >= 2
df_mes_02 = None

if fragmento_02_ativo:
    mes_inicio_02, mes_fim_02 = meses_envelope[1]
    faixas_ptc_mes_02 = [
        (1, 25, "G1"),
        (26, 50, "G2"),
        (51, 75, "G3"),
        (76, 100, "G4"),
    ]
    dfs_db2_02 = []
    for ptc_min, ptc_max, rotulo in faixas_ptc_mes_02:
        sql_grupo = f"""
        SELECT
            SMALLINT(NR_PTC) AS NR_PTC,
            INTEGER(CD_CLI) AS CD_CLI,
            DT_TRAN AS DT_TRAN,
            DECIMAL(VL_TRAN, 15, 2) AS VL_TRAN,
            CHAR(CD_NTZ_CTB_TRAN) AS CD_NTZ_CTB_TRAN,
            INTEGER(CD_CTGR_TRAN_OGNL) AS CD_CTGR_TRAN_OGNL,
            CHAR(CD_TIP_MOE_CRR) AS CD_TIP_MOE_CRR
        FROM DB2GFP.TRAN_RLZD_INST_PCT
        WHERE NR_PTC BETWEEN {ptc_min} AND {ptc_max}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND DT_TRAN >= DATE('{mes_inicio_02}')
          AND DT_TRAN <= DATE('{mes_fim_02}')
        """
        df_g = (
            conector_db2.sql(
                sql_grupo,
                fetchsize=10_000,
                query_timeout=15*60,
                partition_column="NR_PTC",
                lower_bound=ptc_min,
                upper_bound=ptc_max + 1,
                num_partitions=4,
            )
            .drop("NR_PTC")
        )
        dfs_db2_02.append((rotulo, df_g))
    print(f"Mês 02 ({mes_inicio_02} a {mes_fim_02}): 4 blocos x 4 partições JDBC preparados.")
else:
    print("Mês 02 inativo no envelope atual.")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Mês 02 — partições JDBC, sem ação de dados
try:
    if fragmento_02_ativo:
        print("\n[PERF-DIAG] JDBC MÊS 02")
        print("G1:", dfs_db2_02[0][1].rdd.getNumPartitions(), "| G2:", dfs_db2_02[1][1].rdd.getNumPartitions(), "| G3:", dfs_db2_02[2][1].rdd.getNumPartitions(), "| G4:", dfs_db2_02[3][1].rdd.getNumPartitions())
        print("total potencial de fluxos JDBC simultâneos:", sum(item[1].rdd.getNumPartitions() for item in dfs_db2_02))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

if fragmento_02_ativo:
    df_mes_02_bruto = dfs_db2_02[0][1]
    for _, df_grupo in dfs_db2_02[1:]:
        df_mes_02_bruto = df_mes_02_bruto.unionByName(df_grupo)

    df_mes_02_bruto.createOrReplaceTempView("raw_db2_trans_mes_02")

    query_mes_02 = """WITH transacoes_filtradas AS (
    SELECT /*+ BROADCAST(m) */
        t.CD_CLI,
        t.DT_TRAN,
        t.VL_TRAN,
        t.CD_NTZ_CTB_TRAN,
        t.CD_CTGR_TRAN_OGNL,
        t.CD_TIP_MOE_CRR,
        COALESCE(m.CD_CLASS_RADAR, CASE WHEN t.CD_NTZ_CTB_TRAN = 'C' THEN 0 WHEN t.CD_NTZ_CTB_TRAN = 'D' THEN 5 END) AS _CD_CLASS_RADAR,
        COALESCE(m.IN_PARTICIPA, 0) AS _IN_PARTICIPA,
        COALESCE(m.IN_AGRO, 0) AS _IN_AGRO
    FROM raw_db2_trans_mes_02 t
    INNER JOIN vw_janela_financeira j
        ON t.CD_CLI = j.CD_CLI
    LEFT JOIN vw_mapa_categoria_radar m
        ON t.CD_NTZ_CTB_TRAN = m.CD_NTZ_CTB_TRAN
       AND t.CD_CTGR_TRAN_OGNL = m.CD_CTGR_TRAN_OGNL
    WHERE t.DT_TRAN >= j.DT_REF_INI
      AND t.DT_TRAN <= j.DT_REF_FIM
)
SELECT
    CD_CLI,
    COUNT(1) AS QT_TRANS_TOTAL,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_TRANS_ENT,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_TRANS_SAI,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 1 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 2 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 3 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 0 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 4 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_CRED,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 5 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 6 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 7 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 8 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 9 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_OBR,
    CAST(MAX(_IN_AGRO) AS INT) AS IN_AGRO,
    CAST(SUM(CASE WHEN COALESCE(TRIM(CD_TIP_MOE_CRR), '') != 'BRL' THEN 1 ELSE 0 END) AS BIGINT) AS QT_MOE_NAO_BRL
FROM transacoes_filtradas
GROUP BY CD_CLI"""
    df_mes_02 = spark.sql(query_mes_02).persist(StorageLevel.MEMORY_AND_DISK)
    df_mes_02.createOrReplaceTempView("vw_mes_02")

    id_etapa_02 = f"TRANSACOES_MES_02_{mes_inicio_02[:7]}"
    inicio_mes_02 = registrar_etapa(id_etapa_02, "inicio")
    qt_mes_02 = df_mes_02.count()
    registrar_etapa(
        id_etapa_02,
        "fim",
        inicio=inicio_mes_02,
        linhas=qt_mes_02,
        particoes=df_mes_02.rdd.getNumPartitions(),
    )
    dfs_mensais_materializados.append(df_mes_02)
    print(f"Mês 02 materializado uma única vez: {qt_mes_02:,} clientes no agregado mensal.")


### 9.3. Fragmento mensal 03

Leitura DB2 limitada ao recorte real do envelope, 4 blocos × 4 partições JDBC. O agregado mensal é materializado uma única vez e fica disponível para a consolidação final.


In [ ]:
%%spark

fragmento_03_ativo = len(meses_envelope) >= 3
df_mes_03 = None

if fragmento_03_ativo:
    mes_inicio_03, mes_fim_03 = meses_envelope[2]
    faixas_ptc_mes_03 = [
        (1, 25, "G1"),
        (26, 50, "G2"),
        (51, 75, "G3"),
        (76, 100, "G4"),
    ]
    dfs_db2_03 = []
    for ptc_min, ptc_max, rotulo in faixas_ptc_mes_03:
        sql_grupo = f"""
        SELECT
            SMALLINT(NR_PTC) AS NR_PTC,
            INTEGER(CD_CLI) AS CD_CLI,
            DT_TRAN AS DT_TRAN,
            DECIMAL(VL_TRAN, 15, 2) AS VL_TRAN,
            CHAR(CD_NTZ_CTB_TRAN) AS CD_NTZ_CTB_TRAN,
            INTEGER(CD_CTGR_TRAN_OGNL) AS CD_CTGR_TRAN_OGNL,
            CHAR(CD_TIP_MOE_CRR) AS CD_TIP_MOE_CRR
        FROM DB2GFP.TRAN_RLZD_INST_PCT
        WHERE NR_PTC BETWEEN {ptc_min} AND {ptc_max}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND DT_TRAN >= DATE('{mes_inicio_03}')
          AND DT_TRAN <= DATE('{mes_fim_03}')
        """
        df_g = (
            conector_db2.sql(
                sql_grupo,
                fetchsize=10_000,
                query_timeout=15*60,
                partition_column="NR_PTC",
                lower_bound=ptc_min,
                upper_bound=ptc_max + 1,
                num_partitions=4,
            )
            .drop("NR_PTC")
        )
        dfs_db2_03.append((rotulo, df_g))
    print(f"Mês 03 ({mes_inicio_03} a {mes_fim_03}): 4 blocos x 4 partições JDBC preparados.")
else:
    print("Mês 03 inativo no envelope atual.")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Mês 03 — partições JDBC, sem ação de dados
try:
    if fragmento_03_ativo:
        print("\n[PERF-DIAG] JDBC MÊS 03")
        print("G1:", dfs_db2_03[0][1].rdd.getNumPartitions(), "| G2:", dfs_db2_03[1][1].rdd.getNumPartitions(), "| G3:", dfs_db2_03[2][1].rdd.getNumPartitions(), "| G4:", dfs_db2_03[3][1].rdd.getNumPartitions())
        print("total potencial de fluxos JDBC simultâneos:", sum(item[1].rdd.getNumPartitions() for item in dfs_db2_03))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

if fragmento_03_ativo:
    df_mes_03_bruto = dfs_db2_03[0][1]
    for _, df_grupo in dfs_db2_03[1:]:
        df_mes_03_bruto = df_mes_03_bruto.unionByName(df_grupo)

    df_mes_03_bruto.createOrReplaceTempView("raw_db2_trans_mes_03")

    query_mes_03 = """WITH transacoes_filtradas AS (
    SELECT /*+ BROADCAST(m) */
        t.CD_CLI,
        t.DT_TRAN,
        t.VL_TRAN,
        t.CD_NTZ_CTB_TRAN,
        t.CD_CTGR_TRAN_OGNL,
        t.CD_TIP_MOE_CRR,
        COALESCE(m.CD_CLASS_RADAR, CASE WHEN t.CD_NTZ_CTB_TRAN = 'C' THEN 0 WHEN t.CD_NTZ_CTB_TRAN = 'D' THEN 5 END) AS _CD_CLASS_RADAR,
        COALESCE(m.IN_PARTICIPA, 0) AS _IN_PARTICIPA,
        COALESCE(m.IN_AGRO, 0) AS _IN_AGRO
    FROM raw_db2_trans_mes_03 t
    INNER JOIN vw_janela_financeira j
        ON t.CD_CLI = j.CD_CLI
    LEFT JOIN vw_mapa_categoria_radar m
        ON t.CD_NTZ_CTB_TRAN = m.CD_NTZ_CTB_TRAN
       AND t.CD_CTGR_TRAN_OGNL = m.CD_CTGR_TRAN_OGNL
    WHERE t.DT_TRAN >= j.DT_REF_INI
      AND t.DT_TRAN <= j.DT_REF_FIM
)
SELECT
    CD_CLI,
    COUNT(1) AS QT_TRANS_TOTAL,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_TRANS_ENT,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_TRANS_SAI,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 1 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 2 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 3 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 0 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 4 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_CRED,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 5 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 6 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 7 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 8 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 9 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_OBR,
    CAST(MAX(_IN_AGRO) AS INT) AS IN_AGRO,
    CAST(SUM(CASE WHEN COALESCE(TRIM(CD_TIP_MOE_CRR), '') != 'BRL' THEN 1 ELSE 0 END) AS BIGINT) AS QT_MOE_NAO_BRL
FROM transacoes_filtradas
GROUP BY CD_CLI"""
    df_mes_03 = spark.sql(query_mes_03).persist(StorageLevel.MEMORY_AND_DISK)
    df_mes_03.createOrReplaceTempView("vw_mes_03")

    id_etapa_03 = f"TRANSACOES_MES_03_{mes_inicio_03[:7]}"
    inicio_mes_03 = registrar_etapa(id_etapa_03, "inicio")
    qt_mes_03 = df_mes_03.count()
    registrar_etapa(
        id_etapa_03,
        "fim",
        inicio=inicio_mes_03,
        linhas=qt_mes_03,
        particoes=df_mes_03.rdd.getNumPartitions(),
    )
    dfs_mensais_materializados.append(df_mes_03)
    print(f"Mês 03 materializado uma única vez: {qt_mes_03:,} clientes no agregado mensal.")


### 9.4. Fragmento mensal 04

Leitura DB2 limitada ao recorte real do envelope, 4 blocos × 4 partições JDBC. O agregado mensal é materializado uma única vez e fica disponível para a consolidação final.


In [ ]:
%%spark

fragmento_04_ativo = len(meses_envelope) >= 4
df_mes_04 = None

if fragmento_04_ativo:
    mes_inicio_04, mes_fim_04 = meses_envelope[3]
    faixas_ptc_mes_04 = [
        (1, 25, "G1"),
        (26, 50, "G2"),
        (51, 75, "G3"),
        (76, 100, "G4"),
    ]
    dfs_db2_04 = []
    for ptc_min, ptc_max, rotulo in faixas_ptc_mes_04:
        sql_grupo = f"""
        SELECT
            SMALLINT(NR_PTC) AS NR_PTC,
            INTEGER(CD_CLI) AS CD_CLI,
            DT_TRAN AS DT_TRAN,
            DECIMAL(VL_TRAN, 15, 2) AS VL_TRAN,
            CHAR(CD_NTZ_CTB_TRAN) AS CD_NTZ_CTB_TRAN,
            INTEGER(CD_CTGR_TRAN_OGNL) AS CD_CTGR_TRAN_OGNL,
            CHAR(CD_TIP_MOE_CRR) AS CD_TIP_MOE_CRR
        FROM DB2GFP.TRAN_RLZD_INST_PCT
        WHERE NR_PTC BETWEEN {ptc_min} AND {ptc_max}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND DT_TRAN >= DATE('{mes_inicio_04}')
          AND DT_TRAN <= DATE('{mes_fim_04}')
        """
        df_g = (
            conector_db2.sql(
                sql_grupo,
                fetchsize=10_000,
                query_timeout=15*60,
                partition_column="NR_PTC",
                lower_bound=ptc_min,
                upper_bound=ptc_max + 1,
                num_partitions=4,
            )
            .drop("NR_PTC")
        )
        dfs_db2_04.append((rotulo, df_g))
    print(f"Mês 04 ({mes_inicio_04} a {mes_fim_04}): 4 blocos x 4 partições JDBC preparados.")
else:
    print("Mês 04 inativo no envelope atual.")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Mês 04 — partições JDBC, sem ação de dados
try:
    if fragmento_04_ativo:
        print("\n[PERF-DIAG] JDBC MÊS 04")
        print("G1:", dfs_db2_04[0][1].rdd.getNumPartitions(), "| G2:", dfs_db2_04[1][1].rdd.getNumPartitions(), "| G3:", dfs_db2_04[2][1].rdd.getNumPartitions(), "| G4:", dfs_db2_04[3][1].rdd.getNumPartitions())
        print("total potencial de fluxos JDBC simultâneos:", sum(item[1].rdd.getNumPartitions() for item in dfs_db2_04))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

if fragmento_04_ativo:
    df_mes_04_bruto = dfs_db2_04[0][1]
    for _, df_grupo in dfs_db2_04[1:]:
        df_mes_04_bruto = df_mes_04_bruto.unionByName(df_grupo)

    df_mes_04_bruto.createOrReplaceTempView("raw_db2_trans_mes_04")

    query_mes_04 = """WITH transacoes_filtradas AS (
    SELECT /*+ BROADCAST(m) */
        t.CD_CLI,
        t.DT_TRAN,
        t.VL_TRAN,
        t.CD_NTZ_CTB_TRAN,
        t.CD_CTGR_TRAN_OGNL,
        t.CD_TIP_MOE_CRR,
        COALESCE(m.CD_CLASS_RADAR, CASE WHEN t.CD_NTZ_CTB_TRAN = 'C' THEN 0 WHEN t.CD_NTZ_CTB_TRAN = 'D' THEN 5 END) AS _CD_CLASS_RADAR,
        COALESCE(m.IN_PARTICIPA, 0) AS _IN_PARTICIPA,
        COALESCE(m.IN_AGRO, 0) AS _IN_AGRO
    FROM raw_db2_trans_mes_04 t
    INNER JOIN vw_janela_financeira j
        ON t.CD_CLI = j.CD_CLI
    LEFT JOIN vw_mapa_categoria_radar m
        ON t.CD_NTZ_CTB_TRAN = m.CD_NTZ_CTB_TRAN
       AND t.CD_CTGR_TRAN_OGNL = m.CD_CTGR_TRAN_OGNL
    WHERE t.DT_TRAN >= j.DT_REF_INI
      AND t.DT_TRAN <= j.DT_REF_FIM
)
SELECT
    CD_CLI,
    COUNT(1) AS QT_TRANS_TOTAL,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_TRANS_ENT,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_TRANS_SAI,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 1 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 2 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 3 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 0 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 4 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_CRED,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 5 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 6 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 7 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 8 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 9 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_OBR,
    CAST(MAX(_IN_AGRO) AS INT) AS IN_AGRO,
    CAST(SUM(CASE WHEN COALESCE(TRIM(CD_TIP_MOE_CRR), '') != 'BRL' THEN 1 ELSE 0 END) AS BIGINT) AS QT_MOE_NAO_BRL
FROM transacoes_filtradas
GROUP BY CD_CLI"""
    df_mes_04 = spark.sql(query_mes_04).persist(StorageLevel.MEMORY_AND_DISK)
    df_mes_04.createOrReplaceTempView("vw_mes_04")

    id_etapa_04 = f"TRANSACOES_MES_04_{mes_inicio_04[:7]}"
    inicio_mes_04 = registrar_etapa(id_etapa_04, "inicio")
    qt_mes_04 = df_mes_04.count()
    registrar_etapa(
        id_etapa_04,
        "fim",
        inicio=inicio_mes_04,
        linhas=qt_mes_04,
        particoes=df_mes_04.rdd.getNumPartitions(),
    )
    dfs_mensais_materializados.append(df_mes_04)
    print(f"Mês 04 materializado uma única vez: {qt_mes_04:,} clientes no agregado mensal.")


### 9.5. Fragmento mensal 05

Leitura DB2 limitada ao recorte real do envelope, 4 blocos × 4 partições JDBC. O agregado mensal é materializado uma única vez e fica disponível para a consolidação final.


In [ ]:
%%spark

fragmento_05_ativo = len(meses_envelope) >= 5
df_mes_05 = None

if fragmento_05_ativo:
    mes_inicio_05, mes_fim_05 = meses_envelope[4]
    faixas_ptc_mes_05 = [
        (1, 25, "G1"),
        (26, 50, "G2"),
        (51, 75, "G3"),
        (76, 100, "G4"),
    ]
    dfs_db2_05 = []
    for ptc_min, ptc_max, rotulo in faixas_ptc_mes_05:
        sql_grupo = f"""
        SELECT
            SMALLINT(NR_PTC) AS NR_PTC,
            INTEGER(CD_CLI) AS CD_CLI,
            DT_TRAN AS DT_TRAN,
            DECIMAL(VL_TRAN, 15, 2) AS VL_TRAN,
            CHAR(CD_NTZ_CTB_TRAN) AS CD_NTZ_CTB_TRAN,
            INTEGER(CD_CTGR_TRAN_OGNL) AS CD_CTGR_TRAN_OGNL,
            CHAR(CD_TIP_MOE_CRR) AS CD_TIP_MOE_CRR
        FROM DB2GFP.TRAN_RLZD_INST_PCT
        WHERE NR_PTC BETWEEN {ptc_min} AND {ptc_max}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND DT_TRAN >= DATE('{mes_inicio_05}')
          AND DT_TRAN <= DATE('{mes_fim_05}')
        """
        df_g = (
            conector_db2.sql(
                sql_grupo,
                fetchsize=10_000,
                query_timeout=15*60,
                partition_column="NR_PTC",
                lower_bound=ptc_min,
                upper_bound=ptc_max + 1,
                num_partitions=4,
            )
            .drop("NR_PTC")
        )
        dfs_db2_05.append((rotulo, df_g))
    print(f"Mês 05 ({mes_inicio_05} a {mes_fim_05}): 4 blocos x 4 partições JDBC preparados.")
else:
    print("Mês 05 inativo no envelope atual.")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Mês 05 — partições JDBC, sem ação de dados
try:
    if fragmento_05_ativo:
        print("\n[PERF-DIAG] JDBC MÊS 05")
        print("G1:", dfs_db2_05[0][1].rdd.getNumPartitions(), "| G2:", dfs_db2_05[1][1].rdd.getNumPartitions(), "| G3:", dfs_db2_05[2][1].rdd.getNumPartitions(), "| G4:", dfs_db2_05[3][1].rdd.getNumPartitions())
        print("total potencial de fluxos JDBC simultâneos:", sum(item[1].rdd.getNumPartitions() for item in dfs_db2_05))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

if fragmento_05_ativo:
    df_mes_05_bruto = dfs_db2_05[0][1]
    for _, df_grupo in dfs_db2_05[1:]:
        df_mes_05_bruto = df_mes_05_bruto.unionByName(df_grupo)

    df_mes_05_bruto.createOrReplaceTempView("raw_db2_trans_mes_05")

    query_mes_05 = """WITH transacoes_filtradas AS (
    SELECT /*+ BROADCAST(m) */
        t.CD_CLI,
        t.DT_TRAN,
        t.VL_TRAN,
        t.CD_NTZ_CTB_TRAN,
        t.CD_CTGR_TRAN_OGNL,
        t.CD_TIP_MOE_CRR,
        COALESCE(m.CD_CLASS_RADAR, CASE WHEN t.CD_NTZ_CTB_TRAN = 'C' THEN 0 WHEN t.CD_NTZ_CTB_TRAN = 'D' THEN 5 END) AS _CD_CLASS_RADAR,
        COALESCE(m.IN_PARTICIPA, 0) AS _IN_PARTICIPA,
        COALESCE(m.IN_AGRO, 0) AS _IN_AGRO
    FROM raw_db2_trans_mes_05 t
    INNER JOIN vw_janela_financeira j
        ON t.CD_CLI = j.CD_CLI
    LEFT JOIN vw_mapa_categoria_radar m
        ON t.CD_NTZ_CTB_TRAN = m.CD_NTZ_CTB_TRAN
       AND t.CD_CTGR_TRAN_OGNL = m.CD_CTGR_TRAN_OGNL
    WHERE t.DT_TRAN >= j.DT_REF_INI
      AND t.DT_TRAN <= j.DT_REF_FIM
)
SELECT
    CD_CLI,
    COUNT(1) AS QT_TRANS_TOTAL,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_TRANS_ENT,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_TRANS_SAI,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 1 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 2 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 3 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 0 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 4 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_CRED,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 5 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 6 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 7 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 8 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 9 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_OBR,
    CAST(MAX(_IN_AGRO) AS INT) AS IN_AGRO,
    CAST(SUM(CASE WHEN COALESCE(TRIM(CD_TIP_MOE_CRR), '') != 'BRL' THEN 1 ELSE 0 END) AS BIGINT) AS QT_MOE_NAO_BRL
FROM transacoes_filtradas
GROUP BY CD_CLI"""
    df_mes_05 = spark.sql(query_mes_05).persist(StorageLevel.MEMORY_AND_DISK)
    df_mes_05.createOrReplaceTempView("vw_mes_05")

    id_etapa_05 = f"TRANSACOES_MES_05_{mes_inicio_05[:7]}"
    inicio_mes_05 = registrar_etapa(id_etapa_05, "inicio")
    qt_mes_05 = df_mes_05.count()
    registrar_etapa(
        id_etapa_05,
        "fim",
        inicio=inicio_mes_05,
        linhas=qt_mes_05,
        particoes=df_mes_05.rdd.getNumPartitions(),
    )
    dfs_mensais_materializados.append(df_mes_05)
    print(f"Mês 05 materializado uma única vez: {qt_mes_05:,} clientes no agregado mensal.")


### 9.6. Fragmento mensal 06

Leitura DB2 limitada ao recorte real do envelope, 4 blocos × 4 partições JDBC. O agregado mensal é materializado uma única vez e fica disponível para a consolidação final.


In [ ]:
%%spark

fragmento_06_ativo = len(meses_envelope) >= 6
df_mes_06 = None

if fragmento_06_ativo:
    mes_inicio_06, mes_fim_06 = meses_envelope[5]
    faixas_ptc_mes_06 = [
        (1, 25, "G1"),
        (26, 50, "G2"),
        (51, 75, "G3"),
        (76, 100, "G4"),
    ]
    dfs_db2_06 = []
    for ptc_min, ptc_max, rotulo in faixas_ptc_mes_06:
        sql_grupo = f"""
        SELECT
            SMALLINT(NR_PTC) AS NR_PTC,
            INTEGER(CD_CLI) AS CD_CLI,
            DT_TRAN AS DT_TRAN,
            DECIMAL(VL_TRAN, 15, 2) AS VL_TRAN,
            CHAR(CD_NTZ_CTB_TRAN) AS CD_NTZ_CTB_TRAN,
            INTEGER(CD_CTGR_TRAN_OGNL) AS CD_CTGR_TRAN_OGNL,
            CHAR(CD_TIP_MOE_CRR) AS CD_TIP_MOE_CRR
        FROM DB2GFP.TRAN_RLZD_INST_PCT
        WHERE NR_PTC BETWEEN {ptc_min} AND {ptc_max}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND DT_TRAN >= DATE('{mes_inicio_06}')
          AND DT_TRAN <= DATE('{mes_fim_06}')
        """
        df_g = (
            conector_db2.sql(
                sql_grupo,
                fetchsize=10_000,
                query_timeout=15*60,
                partition_column="NR_PTC",
                lower_bound=ptc_min,
                upper_bound=ptc_max + 1,
                num_partitions=4,
            )
            .drop("NR_PTC")
        )
        dfs_db2_06.append((rotulo, df_g))
    print(f"Mês 06 ({mes_inicio_06} a {mes_fim_06}): 4 blocos x 4 partições JDBC preparados.")
else:
    print("Mês 06 inativo no envelope atual.")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Mês 06 — partições JDBC, sem ação de dados
try:
    if fragmento_06_ativo:
        print("\n[PERF-DIAG] JDBC MÊS 06")
        print("G1:", dfs_db2_06[0][1].rdd.getNumPartitions(), "| G2:", dfs_db2_06[1][1].rdd.getNumPartitions(), "| G3:", dfs_db2_06[2][1].rdd.getNumPartitions(), "| G4:", dfs_db2_06[3][1].rdd.getNumPartitions())
        print("total potencial de fluxos JDBC simultâneos:", sum(item[1].rdd.getNumPartitions() for item in dfs_db2_06))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

if fragmento_06_ativo:
    df_mes_06_bruto = dfs_db2_06[0][1]
    for _, df_grupo in dfs_db2_06[1:]:
        df_mes_06_bruto = df_mes_06_bruto.unionByName(df_grupo)

    df_mes_06_bruto.createOrReplaceTempView("raw_db2_trans_mes_06")

    query_mes_06 = """WITH transacoes_filtradas AS (
    SELECT /*+ BROADCAST(m) */
        t.CD_CLI,
        t.DT_TRAN,
        t.VL_TRAN,
        t.CD_NTZ_CTB_TRAN,
        t.CD_CTGR_TRAN_OGNL,
        t.CD_TIP_MOE_CRR,
        COALESCE(m.CD_CLASS_RADAR, CASE WHEN t.CD_NTZ_CTB_TRAN = 'C' THEN 0 WHEN t.CD_NTZ_CTB_TRAN = 'D' THEN 5 END) AS _CD_CLASS_RADAR,
        COALESCE(m.IN_PARTICIPA, 0) AS _IN_PARTICIPA,
        COALESCE(m.IN_AGRO, 0) AS _IN_AGRO
    FROM raw_db2_trans_mes_06 t
    INNER JOIN vw_janela_financeira j
        ON t.CD_CLI = j.CD_CLI
    LEFT JOIN vw_mapa_categoria_radar m
        ON t.CD_NTZ_CTB_TRAN = m.CD_NTZ_CTB_TRAN
       AND t.CD_CTGR_TRAN_OGNL = m.CD_CTGR_TRAN_OGNL
    WHERE t.DT_TRAN >= j.DT_REF_INI
      AND t.DT_TRAN <= j.DT_REF_FIM
)
SELECT
    CD_CLI,
    COUNT(1) AS QT_TRANS_TOTAL,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_TRANS_ENT,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_TRANS_SAI,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 1 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 2 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 3 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 0 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 4 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_CRED,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 5 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 6 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 7 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 8 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 9 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_OBR,
    CAST(MAX(_IN_AGRO) AS INT) AS IN_AGRO,
    CAST(SUM(CASE WHEN COALESCE(TRIM(CD_TIP_MOE_CRR), '') != 'BRL' THEN 1 ELSE 0 END) AS BIGINT) AS QT_MOE_NAO_BRL
FROM transacoes_filtradas
GROUP BY CD_CLI"""
    df_mes_06 = spark.sql(query_mes_06).persist(StorageLevel.MEMORY_AND_DISK)
    df_mes_06.createOrReplaceTempView("vw_mes_06")

    id_etapa_06 = f"TRANSACOES_MES_06_{mes_inicio_06[:7]}"
    inicio_mes_06 = registrar_etapa(id_etapa_06, "inicio")
    qt_mes_06 = df_mes_06.count()
    registrar_etapa(
        id_etapa_06,
        "fim",
        inicio=inicio_mes_06,
        linhas=qt_mes_06,
        particoes=df_mes_06.rdd.getNumPartitions(),
    )
    dfs_mensais_materializados.append(df_mes_06)
    print(f"Mês 06 materializado uma única vez: {qt_mes_06:,} clientes no agregado mensal.")


### 9.7. Fragmento mensal 07

Leitura DB2 limitada ao recorte real do envelope, 4 blocos × 4 partições JDBC. O agregado mensal é materializado uma única vez e fica disponível para a consolidação final.


In [ ]:
%%spark

fragmento_07_ativo = len(meses_envelope) >= 7
df_mes_07 = None

if fragmento_07_ativo:
    mes_inicio_07, mes_fim_07 = meses_envelope[6]
    faixas_ptc_mes_07 = [
        (1, 25, "G1"),
        (26, 50, "G2"),
        (51, 75, "G3"),
        (76, 100, "G4"),
    ]
    dfs_db2_07 = []
    for ptc_min, ptc_max, rotulo in faixas_ptc_mes_07:
        sql_grupo = f"""
        SELECT
            SMALLINT(NR_PTC) AS NR_PTC,
            INTEGER(CD_CLI) AS CD_CLI,
            DT_TRAN AS DT_TRAN,
            DECIMAL(VL_TRAN, 15, 2) AS VL_TRAN,
            CHAR(CD_NTZ_CTB_TRAN) AS CD_NTZ_CTB_TRAN,
            INTEGER(CD_CTGR_TRAN_OGNL) AS CD_CTGR_TRAN_OGNL,
            CHAR(CD_TIP_MOE_CRR) AS CD_TIP_MOE_CRR
        FROM DB2GFP.TRAN_RLZD_INST_PCT
        WHERE NR_PTC BETWEEN {ptc_min} AND {ptc_max}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND DT_TRAN >= DATE('{mes_inicio_07}')
          AND DT_TRAN <= DATE('{mes_fim_07}')
        """
        df_g = (
            conector_db2.sql(
                sql_grupo,
                fetchsize=10_000,
                query_timeout=15*60,
                partition_column="NR_PTC",
                lower_bound=ptc_min,
                upper_bound=ptc_max + 1,
                num_partitions=4,
            )
            .drop("NR_PTC")
        )
        dfs_db2_07.append((rotulo, df_g))
    print(f"Mês 07 ({mes_inicio_07} a {mes_fim_07}): 4 blocos x 4 partições JDBC preparados.")
else:
    print("Mês 07 inativo no envelope atual.")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Mês 07 — partições JDBC, sem ação de dados
try:
    if fragmento_07_ativo:
        print("\n[PERF-DIAG] JDBC MÊS 07")
        print("G1:", dfs_db2_07[0][1].rdd.getNumPartitions(), "| G2:", dfs_db2_07[1][1].rdd.getNumPartitions(), "| G3:", dfs_db2_07[2][1].rdd.getNumPartitions(), "| G4:", dfs_db2_07[3][1].rdd.getNumPartitions())
        print("total potencial de fluxos JDBC simultâneos:", sum(item[1].rdd.getNumPartitions() for item in dfs_db2_07))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

if fragmento_07_ativo:
    df_mes_07_bruto = dfs_db2_07[0][1]
    for _, df_grupo in dfs_db2_07[1:]:
        df_mes_07_bruto = df_mes_07_bruto.unionByName(df_grupo)

    df_mes_07_bruto.createOrReplaceTempView("raw_db2_trans_mes_07")

    query_mes_07 = """WITH transacoes_filtradas AS (
    SELECT /*+ BROADCAST(m) */
        t.CD_CLI,
        t.DT_TRAN,
        t.VL_TRAN,
        t.CD_NTZ_CTB_TRAN,
        t.CD_CTGR_TRAN_OGNL,
        t.CD_TIP_MOE_CRR,
        COALESCE(m.CD_CLASS_RADAR, CASE WHEN t.CD_NTZ_CTB_TRAN = 'C' THEN 0 WHEN t.CD_NTZ_CTB_TRAN = 'D' THEN 5 END) AS _CD_CLASS_RADAR,
        COALESCE(m.IN_PARTICIPA, 0) AS _IN_PARTICIPA,
        COALESCE(m.IN_AGRO, 0) AS _IN_AGRO
    FROM raw_db2_trans_mes_07 t
    INNER JOIN vw_janela_financeira j
        ON t.CD_CLI = j.CD_CLI
    LEFT JOIN vw_mapa_categoria_radar m
        ON t.CD_NTZ_CTB_TRAN = m.CD_NTZ_CTB_TRAN
       AND t.CD_CTGR_TRAN_OGNL = m.CD_CTGR_TRAN_OGNL
    WHERE t.DT_TRAN >= j.DT_REF_INI
      AND t.DT_TRAN <= j.DT_REF_FIM
)
SELECT
    CD_CLI,
    COUNT(1) AS QT_TRANS_TOTAL,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_TRANS_ENT,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_TRANS_SAI,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 1 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 2 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 3 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 0 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 4 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_CRED,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 5 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 6 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 7 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 8 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 9 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_OBR,
    CAST(MAX(_IN_AGRO) AS INT) AS IN_AGRO,
    CAST(SUM(CASE WHEN COALESCE(TRIM(CD_TIP_MOE_CRR), '') != 'BRL' THEN 1 ELSE 0 END) AS BIGINT) AS QT_MOE_NAO_BRL
FROM transacoes_filtradas
GROUP BY CD_CLI"""
    df_mes_07 = spark.sql(query_mes_07).persist(StorageLevel.MEMORY_AND_DISK)
    df_mes_07.createOrReplaceTempView("vw_mes_07")

    id_etapa_07 = f"TRANSACOES_MES_07_{mes_inicio_07[:7]}"
    inicio_mes_07 = registrar_etapa(id_etapa_07, "inicio")
    qt_mes_07 = df_mes_07.count()
    registrar_etapa(
        id_etapa_07,
        "fim",
        inicio=inicio_mes_07,
        linhas=qt_mes_07,
        particoes=df_mes_07.rdd.getNumPartitions(),
    )
    dfs_mensais_materializados.append(df_mes_07)
    print(f"Mês 07 materializado uma única vez: {qt_mes_07:,} clientes no agregado mensal.")


### 9.8. Fragmento mensal 08

Leitura DB2 limitada ao recorte real do envelope, 4 blocos × 4 partições JDBC. O agregado mensal é materializado uma única vez e fica disponível para a consolidação final.


In [ ]:
%%spark

fragmento_08_ativo = len(meses_envelope) >= 8
df_mes_08 = None

if fragmento_08_ativo:
    mes_inicio_08, mes_fim_08 = meses_envelope[7]
    faixas_ptc_mes_08 = [
        (1, 25, "G1"),
        (26, 50, "G2"),
        (51, 75, "G3"),
        (76, 100, "G4"),
    ]
    dfs_db2_08 = []
    for ptc_min, ptc_max, rotulo in faixas_ptc_mes_08:
        sql_grupo = f"""
        SELECT
            SMALLINT(NR_PTC) AS NR_PTC,
            INTEGER(CD_CLI) AS CD_CLI,
            DT_TRAN AS DT_TRAN,
            DECIMAL(VL_TRAN, 15, 2) AS VL_TRAN,
            CHAR(CD_NTZ_CTB_TRAN) AS CD_NTZ_CTB_TRAN,
            INTEGER(CD_CTGR_TRAN_OGNL) AS CD_CTGR_TRAN_OGNL,
            CHAR(CD_TIP_MOE_CRR) AS CD_TIP_MOE_CRR
        FROM DB2GFP.TRAN_RLZD_INST_PCT
        WHERE NR_PTC BETWEEN {ptc_min} AND {ptc_max}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND DT_TRAN >= DATE('{mes_inicio_08}')
          AND DT_TRAN <= DATE('{mes_fim_08}')
        """
        df_g = (
            conector_db2.sql(
                sql_grupo,
                fetchsize=10_000,
                query_timeout=15*60,
                partition_column="NR_PTC",
                lower_bound=ptc_min,
                upper_bound=ptc_max + 1,
                num_partitions=4,
            )
            .drop("NR_PTC")
        )
        dfs_db2_08.append((rotulo, df_g))
    print(f"Mês 08 ({mes_inicio_08} a {mes_fim_08}): 4 blocos x 4 partições JDBC preparados.")
else:
    print("Mês 08 inativo no envelope atual.")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Mês 08 — partições JDBC, sem ação de dados
try:
    if fragmento_08_ativo:
        print("\n[PERF-DIAG] JDBC MÊS 08")
        print("G1:", dfs_db2_08[0][1].rdd.getNumPartitions(), "| G2:", dfs_db2_08[1][1].rdd.getNumPartitions(), "| G3:", dfs_db2_08[2][1].rdd.getNumPartitions(), "| G4:", dfs_db2_08[3][1].rdd.getNumPartitions())
        print("total potencial de fluxos JDBC simultâneos:", sum(item[1].rdd.getNumPartitions() for item in dfs_db2_08))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

if fragmento_08_ativo:
    df_mes_08_bruto = dfs_db2_08[0][1]
    for _, df_grupo in dfs_db2_08[1:]:
        df_mes_08_bruto = df_mes_08_bruto.unionByName(df_grupo)

    df_mes_08_bruto.createOrReplaceTempView("raw_db2_trans_mes_08")

    query_mes_08 = """WITH transacoes_filtradas AS (
    SELECT /*+ BROADCAST(m) */
        t.CD_CLI,
        t.DT_TRAN,
        t.VL_TRAN,
        t.CD_NTZ_CTB_TRAN,
        t.CD_CTGR_TRAN_OGNL,
        t.CD_TIP_MOE_CRR,
        COALESCE(m.CD_CLASS_RADAR, CASE WHEN t.CD_NTZ_CTB_TRAN = 'C' THEN 0 WHEN t.CD_NTZ_CTB_TRAN = 'D' THEN 5 END) AS _CD_CLASS_RADAR,
        COALESCE(m.IN_PARTICIPA, 0) AS _IN_PARTICIPA,
        COALESCE(m.IN_AGRO, 0) AS _IN_AGRO
    FROM raw_db2_trans_mes_08 t
    INNER JOIN vw_janela_financeira j
        ON t.CD_CLI = j.CD_CLI
    LEFT JOIN vw_mapa_categoria_radar m
        ON t.CD_NTZ_CTB_TRAN = m.CD_NTZ_CTB_TRAN
       AND t.CD_CTGR_TRAN_OGNL = m.CD_CTGR_TRAN_OGNL
    WHERE t.DT_TRAN >= j.DT_REF_INI
      AND t.DT_TRAN <= j.DT_REF_FIM
)
SELECT
    CD_CLI,
    COUNT(1) AS QT_TRANS_TOTAL,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_TRANS_ENT,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_TRANS_SAI,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 1 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 2 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 3 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 0 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 4 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_CRED,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 5 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 6 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 7 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 8 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 9 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_OBR,
    CAST(MAX(_IN_AGRO) AS INT) AS IN_AGRO,
    CAST(SUM(CASE WHEN COALESCE(TRIM(CD_TIP_MOE_CRR), '') != 'BRL' THEN 1 ELSE 0 END) AS BIGINT) AS QT_MOE_NAO_BRL
FROM transacoes_filtradas
GROUP BY CD_CLI"""
    df_mes_08 = spark.sql(query_mes_08).persist(StorageLevel.MEMORY_AND_DISK)
    df_mes_08.createOrReplaceTempView("vw_mes_08")

    id_etapa_08 = f"TRANSACOES_MES_08_{mes_inicio_08[:7]}"
    inicio_mes_08 = registrar_etapa(id_etapa_08, "inicio")
    qt_mes_08 = df_mes_08.count()
    registrar_etapa(
        id_etapa_08,
        "fim",
        inicio=inicio_mes_08,
        linhas=qt_mes_08,
        particoes=df_mes_08.rdd.getNumPartitions(),
    )
    dfs_mensais_materializados.append(df_mes_08)
    print(f"Mês 08 materializado uma única vez: {qt_mes_08:,} clientes no agregado mensal.")


### 9.9. Fragmento mensal 09

Leitura DB2 limitada ao recorte real do envelope, 4 blocos × 4 partições JDBC. O agregado mensal é materializado uma única vez e fica disponível para a consolidação final.


In [ ]:
%%spark

fragmento_09_ativo = len(meses_envelope) >= 9
df_mes_09 = None

if fragmento_09_ativo:
    mes_inicio_09, mes_fim_09 = meses_envelope[8]
    faixas_ptc_mes_09 = [
        (1, 25, "G1"),
        (26, 50, "G2"),
        (51, 75, "G3"),
        (76, 100, "G4"),
    ]
    dfs_db2_09 = []
    for ptc_min, ptc_max, rotulo in faixas_ptc_mes_09:
        sql_grupo = f"""
        SELECT
            SMALLINT(NR_PTC) AS NR_PTC,
            INTEGER(CD_CLI) AS CD_CLI,
            DT_TRAN AS DT_TRAN,
            DECIMAL(VL_TRAN, 15, 2) AS VL_TRAN,
            CHAR(CD_NTZ_CTB_TRAN) AS CD_NTZ_CTB_TRAN,
            INTEGER(CD_CTGR_TRAN_OGNL) AS CD_CTGR_TRAN_OGNL,
            CHAR(CD_TIP_MOE_CRR) AS CD_TIP_MOE_CRR
        FROM DB2GFP.TRAN_RLZD_INST_PCT
        WHERE NR_PTC BETWEEN {ptc_min} AND {ptc_max}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND DT_TRAN >= DATE('{mes_inicio_09}')
          AND DT_TRAN <= DATE('{mes_fim_09}')
        """
        df_g = (
            conector_db2.sql(
                sql_grupo,
                fetchsize=10_000,
                query_timeout=15*60,
                partition_column="NR_PTC",
                lower_bound=ptc_min,
                upper_bound=ptc_max + 1,
                num_partitions=4,
            )
            .drop("NR_PTC")
        )
        dfs_db2_09.append((rotulo, df_g))
    print(f"Mês 09 ({mes_inicio_09} a {mes_fim_09}): 4 blocos x 4 partições JDBC preparados.")
else:
    print("Mês 09 inativo no envelope atual.")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Mês 09 — partições JDBC, sem ação de dados
try:
    if fragmento_09_ativo:
        print("\n[PERF-DIAG] JDBC MÊS 09")
        print("G1:", dfs_db2_09[0][1].rdd.getNumPartitions(), "| G2:", dfs_db2_09[1][1].rdd.getNumPartitions(), "| G3:", dfs_db2_09[2][1].rdd.getNumPartitions(), "| G4:", dfs_db2_09[3][1].rdd.getNumPartitions())
        print("total potencial de fluxos JDBC simultâneos:", sum(item[1].rdd.getNumPartitions() for item in dfs_db2_09))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


In [ ]:
%%spark

if fragmento_09_ativo:
    df_mes_09_bruto = dfs_db2_09[0][1]
    for _, df_grupo in dfs_db2_09[1:]:
        df_mes_09_bruto = df_mes_09_bruto.unionByName(df_grupo)

    df_mes_09_bruto.createOrReplaceTempView("raw_db2_trans_mes_09")

    query_mes_09 = """WITH transacoes_filtradas AS (
    SELECT /*+ BROADCAST(m) */
        t.CD_CLI,
        t.DT_TRAN,
        t.VL_TRAN,
        t.CD_NTZ_CTB_TRAN,
        t.CD_CTGR_TRAN_OGNL,
        t.CD_TIP_MOE_CRR,
        COALESCE(m.CD_CLASS_RADAR, CASE WHEN t.CD_NTZ_CTB_TRAN = 'C' THEN 0 WHEN t.CD_NTZ_CTB_TRAN = 'D' THEN 5 END) AS _CD_CLASS_RADAR,
        COALESCE(m.IN_PARTICIPA, 0) AS _IN_PARTICIPA,
        COALESCE(m.IN_AGRO, 0) AS _IN_AGRO
    FROM raw_db2_trans_mes_09 t
    INNER JOIN vw_janela_financeira j
        ON t.CD_CLI = j.CD_CLI
    LEFT JOIN vw_mapa_categoria_radar m
        ON t.CD_NTZ_CTB_TRAN = m.CD_NTZ_CTB_TRAN
       AND t.CD_CTGR_TRAN_OGNL = m.CD_CTGR_TRAN_OGNL
    WHERE t.DT_TRAN >= j.DT_REF_INI
      AND t.DT_TRAN <= j.DT_REF_FIM
)
SELECT
    CD_CLI,
    COUNT(1) AS QT_TRANS_TOTAL,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_TRANS_ENT,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_TRANS_SAI,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 1 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 2 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 3 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 0 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 4 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_ENT_CRED,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 5 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 6 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 7 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 8 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(SUM(CASE WHEN _CD_CLASS_RADAR = 9 AND _IN_PARTICIPA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_SAI_OBR,
    CAST(MAX(_IN_AGRO) AS INT) AS IN_AGRO,
    CAST(SUM(CASE WHEN COALESCE(TRIM(CD_TIP_MOE_CRR), '') != 'BRL' THEN 1 ELSE 0 END) AS BIGINT) AS QT_MOE_NAO_BRL
FROM transacoes_filtradas
GROUP BY CD_CLI"""
    df_mes_09 = spark.sql(query_mes_09).persist(StorageLevel.MEMORY_AND_DISK)
    df_mes_09.createOrReplaceTempView("vw_mes_09")

    id_etapa_09 = f"TRANSACOES_MES_09_{mes_inicio_09[:7]}"
    inicio_mes_09 = registrar_etapa(id_etapa_09, "inicio")
    qt_mes_09 = df_mes_09.count()
    registrar_etapa(
        id_etapa_09,
        "fim",
        inicio=inicio_mes_09,
        linhas=qt_mes_09,
        particoes=df_mes_09.rdd.getNumPartitions(),
    )
    dfs_mensais_materializados.append(df_mes_09)
    print(f"Mês 09 materializado uma única vez: {qt_mes_09:,} clientes no agregado mensal.")


### 9.10. Consolidação única dos meses materializados

Os meses já foram executados sequencialmente contra o DB2. Esta etapa trabalha somente sobre os agregados persistidos e executa **um único** agrupamento histórico por cliente.


In [ ]:
%%spark

if not dfs_mensais_materializados:
    raise ValueError("Nenhum fragmento mensal foi materializado.")

df_meses_union = dfs_mensais_materializados[0]
for df_mes_materializado in dfs_mensais_materializados[1:]:
    df_meses_union = df_meses_union.unionByName(df_mes_materializado)

df_meses_union.createOrReplaceTempView("vw_meses_materializados_union")

query_acumulador_final = """
SELECT
    CD_CLI,
    SUM(QT_TRANS_TOTAL) AS QT_TRANS_TOTAL,
    SUM(QT_TRANS_ENT) AS QT_TRANS_ENT,
    SUM(QT_TRANS_SAI) AS QT_TRANS_SAI,
    CAST(SUM(VL_TRANS_ENT) AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(SUM(VL_TRANS_SAI) AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(SUM(VL_ENT_REN) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(SUM(VL_ENT_EST) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(SUM(VL_ENT_RESG) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(SUM(VL_ENT_OUT) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(SUM(VL_ENT_CRED) AS DECIMAL(25,2)) AS VL_ENT_CRED,
    CAST(SUM(VL_SAI_IND) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(SUM(VL_SAI_ESS) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(SUM(VL_SAI_NAO_ESS) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(SUM(VL_SAI_FUT) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(SUM(VL_SAI_OBR) AS DECIMAL(25,2)) AS VL_SAI_OBR,
    CAST(MAX(IN_AGRO) AS INT) AS IN_AGRO,
    SUM(QT_MOE_NAO_BRL) AS QT_MOE_NAO_BRL
FROM vw_meses_materializados_union
GROUP BY CD_CLI
"""

df_acumulador = spark.sql(query_acumulador_final).persist(StorageLevel.MEMORY_AND_DISK)
df_acumulador.createOrReplaceTempView("vw_acumulador")

id_acumulador = "SPARK_ACUM_CONSOLIDACAO_UNICA"
inicio_acumulador = registrar_etapa(id_acumulador, "inicio")
qt_acumulador = df_acumulador.count()
registrar_etapa(
    id_acumulador,
    "fim",
    inicio=inicio_acumulador,
    linhas=qt_acumulador,
    particoes=df_acumulador.rdd.getNumPartitions(),
)

for df_mes_materializado in dfs_mensais_materializados:
    df_mes_materializado.unpersist()

print(f"Acumulador final consolidado UMA VEZ: {qt_acumulador:,} clientes. Agregados mensais intermediários liberados.")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Acumulador final — metadados locais
try:
    print("\n[PERF-DIAG] ACUMULADOR FINAL")
    print("partições:", df_acumulador.rdd.getNumPartitions())
    print("storageLevel:", df_acumulador.storageLevel)
    print("meses materializados consolidados:", len(dfs_mensais_materializados))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


## 10. Resultado transacional consolidado (Spark SQL)

A `vw_agregacoes` agora nasce do acumulador consolidado em uma única passagem, sem reprocessamento recursivo de meses anteriores.


In [ ]:
%%spark

if df_acumulador is None:
    raise ValueError("Nenhum fragmento mensal foi processado.")

query_agregacoes = f"""
SELECT
    CAST(CD_CLI AS INT) AS CD_CLI,
    CAST(QT_TRANS_TOTAL AS BIGINT) AS QT_TRANS_TOTAL,
    CAST(QT_TRANS_ENT AS BIGINT) AS QT_TRANS_ENT,
    CAST(QT_TRANS_SAI AS BIGINT) AS QT_TRANS_SAI,
    CAST(VL_TRANS_ENT AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(VL_TRANS_SAI AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(VL_ENT_REN AS DECIMAL(18,2)) AS VL_ENT_REN,
    CAST(VL_ENT_EST AS DECIMAL(18,2)) AS VL_ENT_EST,
    CAST(VL_ENT_RESG AS DECIMAL(18,2)) AS VL_ENT_RESG,
    CAST(VL_ENT_OUT AS DECIMAL(18,2)) AS VL_ENT_OUT,
    CAST(VL_ENT_CRED AS DECIMAL(18,2)) AS VL_ENT_CRED,
    CAST(VL_SAI_IND AS DECIMAL(18,2)) AS VL_SAI_IND,
    CAST(VL_SAI_ESS AS DECIMAL(18,2)) AS VL_SAI_ESS,
    CAST(VL_SAI_NAO_ESS AS DECIMAL(18,2)) AS VL_SAI_NAO_ESS,
    CAST(VL_SAI_FUT AS DECIMAL(18,2)) AS VL_SAI_FUT,
    CAST(VL_SAI_OBR AS DECIMAL(18,2)) AS VL_SAI_OBR,
    CASE WHEN IN_AGRO = 1 THEN 'S' ELSE 'N' END AS FL_TEM_MOV_AGRO,
    CASE WHEN QT_MOE_NAO_BRL > 0 THEN 'S' ELSE 'N' END AS FL_MULT_MOE
FROM vw_acumulador
"""

df_agregacoes = spark.sql(query_agregacoes)
df_agregacoes.createOrReplaceTempView("vw_agregacoes")
print(f"vw_agregacoes registrada via Spark SQL com {qt_acumulador:,} clientes e colunas de grupos consolidadas.")


In [ ]:
%%spark

id_controle_moeda = "MONITORA_MOEDA"
inicio_controle_moeda = registrar_etapa(id_controle_moeda, "inicio")
controle_moeda = spark.sql("""
    SELECT
        SUM(QT_MOE_NAO_BRL) AS QT_MOE_NAO_BRL,
        SUM(CASE WHEN QT_MOE_NAO_BRL > 0 THEN 1 ELSE 0 END) AS QT_CLI_MULTI_MOE
    FROM vw_acumulador
""").first()
registrar_etapa(id_controle_moeda, "fim", inicio=inicio_controle_moeda, linhas=1)
qt_moeda_nao_brl = int(controle_moeda["QT_MOE_NAO_BRL"] or 0)
qt_cli_multi_moe = int(controle_moeda["QT_CLI_MULTI_MOE"] or 0)
print(f"Monitoramento de moeda: {qt_moeda_nao_brl:,} transações não BRL | {qt_cli_multi_moe:,} clientes marcados com FL_MULT_MOE='S'.")


## 11. Construção modular da tabela analítica (8 Blocos Spark SQL)

Cada bloco adiciona conjuntos específicos de colunas com regras de negócio claras, instrumentação individual e diagnóstico de desempenho 100% em Spark SQL.


### 11.1 Bloco 1: Base Cadastral, Janela e Totais Brutos
- **Colunas adicionadas:** `CD_CLI`, `TS_ATL_TRAN`, `DD_INC_MM_CLC_BLC`, `VL_REN_PRES`, `CD_MAC_PRFL_CLI`, `NM_MAC_PRFL_CLI`, `CD_MIC_PRFL_CLI`, `NM_MIC_PRFL_CLI`, `NM_PRFL_FIN`, `DT_REF_INI`, `DT_REF_FIM`, `DT_EXEA`, `QT_TRANS_TOTAL`, `QT_TRANS_ENT`, `QT_TRANS_SAI`, `VL_TRANS_ENT`, `VL_TRANS_SAI`, `VL_ENT_REN`, `VL_ENT_EST`, `VL_ENT_RESG`, `VL_ENT_OUT`, `VL_ENT_CRED`, `VL_ENT_TOTAL`, `VL_SAI_IND`, `VL_SAI_ESS`, `VL_SAI_NAO_ESS`, `VL_SAI_FUT`, `VL_SAI_OBR`, `VL_SAI_TOTAL`.


In [ ]:
%%spark

query_bloco_01 = f"""
SELECT
    j.CD_CLI,
    j.TS_ATL_TRAN,
    j.DD_INC_MM_CLC_BLC,
    r.VL_REN_PRES,
    p.CD_MAC_PRFL_CLI,
    p.NM_MAC_PRFL_CLI,
    p.CD_MIC_PRFL_CLI,
    p.NM_MIC_PRFL_CLI,
    p.NM_PRFL_FIN,
    j.DT_REF_INI,
    j.DT_REF_FIM,
    CAST(DATE('{data_atual}') AS DATE) AS DT_EXEA,
    COALESCE(a.QT_TRANS_TOTAL, 0) AS QT_TRANS_TOTAL,
    COALESCE(a.QT_TRANS_ENT, 0) AS QT_TRANS_ENT,
    COALESCE(a.QT_TRANS_SAI, 0) AS QT_TRANS_SAI,
    COALESCE(a.VL_TRANS_ENT, CAST(0 AS DECIMAL(25,2))) AS VL_TRANS_ENT,
    COALESCE(a.VL_TRANS_SAI, CAST(0 AS DECIMAL(25,2))) AS VL_TRANS_SAI,
    COALESCE(a.VL_ENT_REN, CAST(0 AS DECIMAL(18,2))) AS VL_ENT_REN,
    COALESCE(a.VL_ENT_EST, CAST(0 AS DECIMAL(18,2))) AS VL_ENT_EST,
    COALESCE(a.VL_ENT_RESG, CAST(0 AS DECIMAL(18,2))) AS VL_ENT_RESG,
    COALESCE(a.VL_ENT_OUT, CAST(0 AS DECIMAL(18,2))) AS VL_ENT_OUT,
    COALESCE(a.VL_ENT_CRED, CAST(0 AS DECIMAL(18,2))) AS VL_ENT_CRED,
    CAST(
        COALESCE(a.VL_ENT_REN, 0) + COALESCE(a.VL_ENT_EST, 0) + COALESCE(a.VL_ENT_RESG, 0) + COALESCE(a.VL_ENT_OUT, 0) + COALESCE(a.VL_ENT_CRED, 0)
        AS DECIMAL(18,2)
    ) AS VL_ENT_TOTAL,
    COALESCE(a.VL_SAI_IND, CAST(0 AS DECIMAL(18,2))) AS VL_SAI_IND,
    COALESCE(a.VL_SAI_ESS, CAST(0 AS DECIMAL(18,2))) AS VL_SAI_ESS,
    COALESCE(a.VL_SAI_NAO_ESS, CAST(0 AS DECIMAL(18,2))) AS VL_SAI_NAO_ESS,
    COALESCE(a.VL_SAI_FUT, CAST(0 AS DECIMAL(18,2))) AS VL_SAI_FUT,
    COALESCE(a.VL_SAI_OBR, CAST(0 AS DECIMAL(18,2))) AS VL_SAI_OBR,
    CAST(
        COALESCE(a.VL_SAI_IND, 0) + COALESCE(a.VL_SAI_ESS, 0) + COALESCE(a.VL_SAI_NAO_ESS, 0) + COALESCE(a.VL_SAI_FUT, 0) + COALESCE(a.VL_SAI_OBR, 0)
        AS DECIMAL(18,2)
    ) AS VL_SAI_TOTAL,
    COALESCE(a.FL_TEM_MOV_AGRO, 'N') AS FL_TEM_MOV_AGRO,
    COALESCE(a.FL_MULT_MOE, 'N') AS FL_MULT_MOE,
    CAST(NULL AS STRING) AS FL_PARTICIPA_RADAR
FROM vw_janela_financeira j
LEFT JOIN vw_renda_presumida r ON j.CD_CLI = r.CD_CLI
LEFT JOIN vw_perfil_financeiro p ON j.CD_CLI = p.CD_CLI
LEFT JOIN vw_agregacoes a ON j.CD_CLI = a.CD_CLI
"""
df_bloco_01 = spark.sql(query_bloco_01)
df_bloco_01.createOrReplaceTempView("vw_bloco_01")

id_bloco_01 = "ANALITICO_BLOCO_01_BASE_TOTAIS"
inicio_bloco_01 = registrar_etapa(id_bloco_01, "inicio")
print(f"Bloco 1 preparado via Spark SQL com grupos de débito. Partições: {df_bloco_01.rdd.getNumPartitions()}")
registrar_etapa(id_bloco_01, "fim", inicio=inicio_bloco_01, particoes=df_bloco_01.rdd.getNumPartitions())


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Primeiro bloco analítico — somente metadados
try:
    print("\n[PERF-DIAG] BLOCO ANALÍTICO 01")
    print("partições planejadas:", df_bloco_01.rdd.getNumPartitions())
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


### 11.2 Bloco 2: Indicadores de Orçamento (Cols 30 a 36)
- **Colunas adicionadas:** `VL_RES_ORC`, `PC_SAI_ENT`, `CD_RES_ORC`, `TX_RES_ORC`, `CD_FAIXA_ORC`, `TX_STS_RES`, `TX_STS_FINAL`.


In [ ]:
%%spark

query_bloco_02 = """
WITH calculo_orcamento AS (
    SELECT
        b.*,
        CAST(
            CASE WHEN QT_TRANS_TOTAL = 0 THEN NULL ELSE VL_ENT_TOTAL - VL_SAI_TOTAL END
            AS DECIMAL(18,2)
        ) AS VL_RES_ORC,
        CAST(
            COALESCE(VL_SAI_TOTAL / NULLIF(VL_ENT_TOTAL, 0), 0)
            AS DECIMAL(9,6)
        ) AS PC_SAI_ENT
    FROM vw_bloco_01 b
),
faixas AS (
    SELECT
        c.*,
        CAST(
            CASE
                WHEN QT_TRANS_TOTAL = 0 THEN NULL
                WHEN PC_SAI_ENT BETWEEN 0.950000 AND 1.050000 THEN 0
                WHEN PC_SAI_ENT > 1.050000 AND PC_SAI_ENT <= 1.250000 THEN 1
                WHEN PC_SAI_ENT > 1.250000 THEN 2
                WHEN PC_SAI_ENT >= 0.750000 AND PC_SAI_ENT < 0.950000 THEN 3
                ELSE 4
            END AS INT
        ) AS CD_FAIXA_ORC
    FROM calculo_orcamento c
)
SELECT
    f.*,
    CAST(
        CASE
            WHEN CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 0 THEN 0
            WHEN CD_FAIXA_ORC IN (3, 4) THEN 1
            ELSE 2
        END AS INT
    ) AS CD_RES_ORC,
    CASE
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
        WHEN CD_FAIXA_ORC IN (3, 4) THEN 'Superavitário'
        ELSE 'Deficitário'
    END AS TX_RES_ORC,
    CASE
        WHEN CD_FAIXA_ORC IS NULL OR CD_FAIXA_ORC = 0 THEN NULL
        WHEN CD_FAIXA_ORC IN (2, 4) THEN 'Acentuado'
        ELSE 'Moderado'
    END AS TX_STS_RES,
    CASE
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
        WHEN CD_FAIXA_ORC = 1 THEN 'Deficitário Moderado'
        WHEN CD_FAIXA_ORC = 2 THEN 'Deficitário Acentuado'
        WHEN CD_FAIXA_ORC = 3 THEN 'Superavitário Moderado'
        ELSE 'Superavitário Acentuado'
    END AS TX_STS_FINAL
FROM faixas f
"""
df_bloco_02 = spark.sql(query_bloco_02)
df_bloco_02.createOrReplaceTempView("vw_bloco_02")

id_bloco_02 = "ANALITICO_BLOCO_02_ORCAMENTO"
inicio_bloco_02 = registrar_etapa(id_bloco_02, "inicio")
print(f"Bloco 2 preparado via Spark SQL (Indicadores de Orçamento). Partições: {df_bloco_02.rdd.getNumPartitions()}")
registrar_etapa(id_bloco_02, "fim", inicio=inicio_bloco_02, particoes=df_bloco_02.rdd.getNumPartitions())


### 11.3 Bloco 3: Percentuais sobre Renda e Parâmetros de Referência (Cols 37 a 46)
- **Colunas adicionadas:** `PC_SAI_IND`, `PC_SAI_ESS`, `PC_SAI_NAO_ESS`, `PC_SAI_FUT`, `PC_SAI_OBR`, `PC_REF_IND`, `PC_REF_ESS`, `PC_REF_NAO_ESS`, `PC_REF_FUT`, `PC_REF_OBR`.


In [ ]:
%%spark

query_bloco_03 = """
SELECT
    o.*,
    CAST(
        CASE WHEN VL_REN_PRES > 0 THEN COALESCE(VL_SAI_IND / VL_REN_PRES, 0) ELSE 0.000000 END
        AS DECIMAL(9,6)
    ) AS PC_SAI_IND,
    CAST(
        CASE WHEN VL_REN_PRES > 0 THEN COALESCE(VL_SAI_ESS / VL_REN_PRES, 0) ELSE 0.000000 END
        AS DECIMAL(9,6)
    ) AS PC_SAI_ESS,
    CAST(
        CASE WHEN VL_REN_PRES > 0 THEN COALESCE(VL_SAI_NAO_ESS / VL_REN_PRES, 0) ELSE 0.000000 END
        AS DECIMAL(9,6)
    ) AS PC_SAI_NAO_ESS,
    CAST(
        CASE WHEN VL_REN_PRES > 0 THEN COALESCE(VL_SAI_FUT / VL_REN_PRES, 0) ELSE 0.000000 END
        AS DECIMAL(9,6)
    ) AS PC_SAI_FUT,
    CAST(
        CASE WHEN VL_REN_PRES > 0 THEN COALESCE(VL_SAI_OBR / VL_REN_PRES, 0) ELSE 0.000000 END
        AS DECIMAL(9,6)
    ) AS PC_SAI_OBR,
    CAST(0.750000 AS DECIMAL(9,6)) AS PC_REF_IND,
    CAST(0.500000 AS DECIMAL(9,6)) AS PC_REF_ESS,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_NAO_ESS,
    CAST(0.200000 AS DECIMAL(9,6)) AS PC_REF_FUT,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_OBR
FROM vw_bloco_02 o
"""
df_bloco_03 = spark.sql(query_bloco_03)
df_bloco_03.createOrReplaceTempView("vw_bloco_03")

id_bloco_03 = "ANALITICO_BLOCO_03_PERCENTUAIS_REFERENCIAS"
inicio_bloco_03 = registrar_etapa(id_bloco_03, "inicio")
print(f"Bloco 3 preparado via Spark SQL (Percentuais e Parâmetros). Partições: {df_bloco_03.rdd.getNumPartitions()}")
registrar_etapa(id_bloco_03, "fim", inicio=inicio_bloco_03, particoes=df_bloco_03.rdd.getNumPartitions())


### 11.4 Bloco 4: Pontuação por Concentração (Cols 47 a 51)
- **Colunas adicionadas:** `NR_PONT_CONC_IND`, `NR_PONT_CONC_ESS`, `NR_PONT_CONC_NAO_ESS`, `NR_PONT_CONC_FUT`, `NR_PONT_CONC_OBR`.


In [ ]:
%%spark

query_bloco_04 = """
SELECT
    p.*,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_IND > PC_REF_IND THEN 99
            ELSE 0
        END AS INT
    ) AS NR_PONT_CONC_IND,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_ESS < PC_REF_ESS THEN 0
            WHEN PC_SAI_ESS < PC_REF_ESS * 1.5 THEN 1
            ELSE 2
        END AS INT
    ) AS NR_PONT_CONC_ESS,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_NAO_ESS < PC_REF_NAO_ESS THEN 0
            WHEN PC_SAI_NAO_ESS < PC_REF_NAO_ESS * 1.5 THEN 1
            ELSE 2
        END AS INT
    ) AS NR_PONT_CONC_NAO_ESS,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_FUT >= PC_REF_FUT * 1.5 THEN 0
            WHEN PC_SAI_FUT >= PC_REF_FUT THEN 1
            ELSE 2
        END AS INT
    ) AS NR_PONT_CONC_FUT,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_OBR < PC_REF_OBR THEN 0
            WHEN PC_SAI_OBR < PC_REF_OBR * 1.5 THEN 1
            ELSE 2
        END AS INT
    ) AS NR_PONT_CONC_OBR
FROM vw_bloco_03 p
"""
df_bloco_04 = spark.sql(query_bloco_04)
df_bloco_04.createOrReplaceTempView("vw_bloco_04")

id_bloco_04 = "ANALITICO_BLOCO_04_PONTUACAO_CONCENTRACAO"
inicio_bloco_04 = registrar_etapa(id_bloco_04, "inicio")
print(f"Bloco 4 preparado via Spark SQL (Pontuação por Concentração). Partições: {df_bloco_04.rdd.getNumPartitions()}")
registrar_etapa(id_bloco_04, "fim", inicio=inicio_bloco_04, particoes=df_bloco_04.rdd.getNumPartitions())


### 11.5 Bloco 5: Pontuação Orçamentária (Cols 52 a 56)
- **Colunas adicionadas:** `NR_PONT_ORC_IND`, `NR_PONT_ORC_ESS`, `NR_PONT_ORC_NAO_ESS`, `NR_PONT_ORC_FUT`, `NR_PONT_ORC_OBR`.


In [ ]:
%%spark

query_bloco_05 = """
SELECT
    c.*,
    CAST(
        CASE WHEN QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END
        AS INT
    ) AS NR_PONT_ORC_IND,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_ORC_ESS,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_ORC_NAO_ESS,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 4 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 3) THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_ORC_FUT,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_ORC_OBR
FROM vw_bloco_04 c
"""
df_bloco_05 = spark.sql(query_bloco_05)
df_bloco_05.createOrReplaceTempView("vw_bloco_05")

id_bloco_05 = "ANALITICO_BLOCO_05_PONTUACAO_ORCAMENTO"
inicio_bloco_05 = registrar_etapa(id_bloco_05, "inicio")
print(f"Bloco 5 preparado via Spark SQL (Pontuação Orçamentária). Partições: {df_bloco_05.rdd.getNumPartitions()}")
registrar_etapa(id_bloco_05, "fim", inicio=inicio_bloco_05, particoes=df_bloco_05.rdd.getNumPartitions())


### 11.6 Bloco 6: Pontuação por Perfil (Cols 57 a 61)
- **Colunas adicionadas:** `NR_PONT_PRFL_IND`, `NR_PONT_PRFL_ESS`, `NR_PONT_PRFL_NAO_ESS`, `NR_PONT_PRFL_FUT`, `NR_PONT_PRFL_OBR`.


In [ ]:
%%spark

query_bloco_06 = """
SELECT
    o.*,
    CAST(
        CASE WHEN QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END
        AS INT
    ) AS NR_PONT_PRFL_IND,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 OR NM_PRFL_FIN IS NULL OR NM_PRFL_FIN = 'CODIGO NAO MAPEADO' THEN NULL
            WHEN NM_PRFL_FIN = 'Endividado Acrobata' THEN 2
            WHEN NM_PRFL_FIN = 'Endividado Inadimplente'
              OR NM_PRFL_FIN IN ('Equilibrista', 'Investidor Precavido', 'Investidor Despreocupado', 'Investidor Acelerado') THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_PRFL_ESS,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 OR NM_PRFL_FIN IS NULL OR NM_PRFL_FIN = 'CODIGO NAO MAPEADO' THEN NULL
            WHEN NM_PRFL_FIN = 'Endividado Consciente' THEN 2
            WHEN NM_PRFL_FIN = 'Endividado Iminente' THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_PRFL_NAO_ESS,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 OR NM_PRFL_FIN IS NULL OR NM_PRFL_FIN = 'CODIGO NAO MAPEADO' THEN NULL
            WHEN NM_PRFL_FIN IN ('Equilibrista', 'Investidor Precavido', 'Investidor Despreocupado', 'Investidor Acelerado') THEN 2
            WHEN NM_PRFL_FIN = 'Endividado Consciente' THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_PRFL_FUT,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 OR NM_PRFL_FIN IS NULL OR NM_PRFL_FIN = 'CODIGO NAO MAPEADO' THEN NULL
            WHEN NM_PRFL_FIN IN ('Endividado Iminente', 'Endividado Inadimplente') THEN 2
            WHEN NM_PRFL_FIN = 'Endividado Acrobata' THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_PRFL_OBR
FROM vw_bloco_05 o
"""
df_bloco_06 = spark.sql(query_bloco_06)
df_bloco_06.createOrReplaceTempView("vw_bloco_06")

id_bloco_06 = "ANALITICO_BLOCO_06_PONTUACAO_PERFIL"
inicio_bloco_06 = registrar_etapa(id_bloco_06, "inicio")
print(f"Bloco 6 preparado via Spark SQL (Pontuação por Perfil). Partições: {df_bloco_06.rdd.getNumPartitions()}")
registrar_etapa(id_bloco_06, "fim", inicio=inicio_bloco_06, particoes=df_bloco_06.rdd.getNumPartitions())


### 11.7 Bloco 7: Pontuação Consolidada e Tema Vencedor (Cols 62 a 68)
- **Colunas adicionadas:** `NR_PONT_IND_FIM`, `NR_PONT_ESS_FIM`, `NR_PONT_NAO_ESS_FIM`, `NR_PONT_FUT_FIM`, `NR_PONT_OBR_FIM`, `CD_TEMA_VENCEDOR`, `TX_TEMA_VENCEDOR`.


In [ ]:
%%spark

query_bloco_07 = """
WITH consolidacao AS (
    SELECT
        p.*,
        CAST(NR_PONT_CONC_IND AS INT) AS NR_PONT_IND_FIM,
        CAST(
            CASE
                WHEN NR_PONT_CONC_ESS IS NULL OR NR_PONT_ORC_ESS IS NULL OR NR_PONT_PRFL_ESS IS NULL THEN NULL
                ELSE NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS
            END AS INT
        ) AS NR_PONT_ESS_FIM,
        CAST(
            CASE
                WHEN NR_PONT_CONC_NAO_ESS IS NULL OR NR_PONT_ORC_NAO_ESS IS NULL OR NR_PONT_PRFL_NAO_ESS IS NULL THEN NULL
                ELSE NR_PONT_CONC_NAO_ESS + NR_PONT_ORC_NAO_ESS + NR_PONT_PRFL_NAO_ESS
            END AS INT
        ) AS NR_PONT_NAO_ESS_FIM,
        CAST(
            CASE
                WHEN NR_PONT_CONC_FUT IS NULL OR NR_PONT_ORC_FUT IS NULL OR NR_PONT_PRFL_FUT IS NULL THEN NULL
                ELSE NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT
            END AS INT
        ) AS NR_PONT_FUT_FIM,
        CAST(
            CASE
                WHEN NR_PONT_CONC_OBR IS NULL OR NR_PONT_ORC_OBR IS NULL OR NR_PONT_PRFL_OBR IS NULL THEN NULL
                ELSE NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR
            END AS INT
        ) AS NR_PONT_OBR_FIM
    FROM vw_bloco_06 p
),
maximos AS (
    SELECT
        c.*,
        CASE
            WHEN NR_PONT_IND_FIM IS NULL OR NR_PONT_ESS_FIM IS NULL OR NR_PONT_NAO_ESS_FIM IS NULL OR NR_PONT_FUT_FIM IS NULL OR NR_PONT_OBR_FIM IS NULL THEN NULL
            ELSE GREATEST(NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM, NR_PONT_FUT_FIM, NR_PONT_OBR_FIM)
        END AS NR_PONT_MAX
    FROM consolidacao c
),
decisao_vencedor AS (
    SELECT
        m.*,
        CASE
            WHEN NR_PONT_MAX IS NULL THEN NULL
            WHEN CAST(NR_PONT_IND_FIM = NR_PONT_MAX AS INT)
               + CAST(NR_PONT_ESS_FIM = NR_PONT_MAX AS INT)
               + CAST(NR_PONT_NAO_ESS_FIM = NR_PONT_MAX AS INT)
               + CAST(NR_PONT_FUT_FIM = NR_PONT_MAX AS INT)
               + CAST(NR_PONT_OBR_FIM = NR_PONT_MAX AS INT) > 1 THEN 9
            WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 1
            WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 2
            WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 3
            WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 4
            ELSE 5
        END AS CD_TEMA_VENCEDOR
    FROM maximos m
)
SELECT
    d.*,
    CASE CD_TEMA_VENCEDOR
        WHEN 1 THEN 'Categorização dos Gastos'
        WHEN 2 THEN 'Gestão de Orçamento'
        WHEN 3 THEN 'Consumo Planejado'
        WHEN 4 THEN 'Formação de Reserva'
        WHEN 5 THEN 'Uso Consciente do Crédito'
        WHEN 9 THEN 'Empate'
    END AS TX_TEMA_VENCEDOR
FROM decisao_vencedor d
"""
df_bloco_07 = spark.sql(query_bloco_07)
df_bloco_07.createOrReplaceTempView("vw_bloco_07")

id_bloco_07 = "ANALITICO_BLOCO_07_PONTUACAO_FINAL_VENCEDOR"
inicio_bloco_07 = registrar_etapa(id_bloco_07, "inicio")
print(f"Bloco 7 preparado via Spark SQL (Pontuação Consolidada e Vencedor). Partições: {df_bloco_07.rdd.getNumPartitions()}")
registrar_etapa(id_bloco_07, "fim", inicio=inicio_bloco_07, particoes=df_bloco_07.rdd.getNumPartitions())


### 11.8 Bloco 8: Projeção Canônica (72 Colunas Físicas)
- **Colunas finais:** 71 campos analíticos base + coluna de partição `DT_MES_EXEA`.


In [ ]:
%%spark

query_bloco_08 = f"""
SELECT
    CAST(CD_CLI AS INT) AS CD_CLI,
    CAST(TS_ATL_TRAN AS TIMESTAMP) AS TS_ATL_TRAN,
    CAST(DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC,
    CAST(VL_REN_PRES AS DECIMAL(18,2)) AS VL_REN_PRES,
    CAST(CD_MAC_PRFL_CLI AS BIGINT) AS CD_MAC_PRFL_CLI,
    CAST(NM_MAC_PRFL_CLI AS STRING) AS NM_MAC_PRFL_CLI,
    CAST(CD_MIC_PRFL_CLI AS BIGINT) AS CD_MIC_PRFL_CLI,
    CAST(NM_MIC_PRFL_CLI AS STRING) AS NM_MIC_PRFL_CLI,
    CAST(NM_PRFL_FIN AS STRING) AS NM_PRFL_FIN,
    CAST(DT_REF_INI AS DATE) AS DT_REF_INI,
    CAST(DT_REF_FIM AS DATE) AS DT_REF_FIM,
    CAST(DATE('{data_atual}') AS DATE) AS DT_EXEA,
    CAST(QT_TRANS_TOTAL AS BIGINT) AS QT_TRANS_TOTAL,
    CAST(QT_TRANS_ENT AS BIGINT) AS QT_TRANS_ENT,
    CAST(QT_TRANS_SAI AS BIGINT) AS QT_TRANS_SAI,
    CAST(VL_TRANS_ENT AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(VL_TRANS_SAI AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(VL_ENT_REN AS DECIMAL(18,2)) AS VL_ENT_REN,
    CAST(VL_ENT_EST AS DECIMAL(18,2)) AS VL_ENT_EST,
    CAST(VL_ENT_RESG AS DECIMAL(18,2)) AS VL_ENT_RESG,
    CAST(VL_ENT_OUT AS DECIMAL(18,2)) AS VL_ENT_OUT,
    CAST(VL_ENT_CRED AS DECIMAL(18,2)) AS VL_ENT_CRED,
    CAST(VL_ENT_TOTAL AS DECIMAL(18,2)) AS VL_ENT_TOTAL,
    CAST(VL_SAI_IND AS DECIMAL(18,2)) AS VL_SAI_IND,
    CAST(VL_SAI_ESS AS DECIMAL(18,2)) AS VL_SAI_ESS,
    CAST(VL_SAI_NAO_ESS AS DECIMAL(18,2)) AS VL_SAI_NAO_ESS,
    CAST(VL_SAI_FUT AS DECIMAL(18,2)) AS VL_SAI_FUT,
    CAST(VL_SAI_OBR AS DECIMAL(18,2)) AS VL_SAI_OBR,
    CAST(VL_SAI_TOTAL AS DECIMAL(18,2)) AS VL_SAI_TOTAL,
    CAST(VL_RES_ORC AS DECIMAL(18,2)) AS VL_RES_ORC,
    CAST(PC_SAI_ENT AS DECIMAL(9,6)) AS PC_SAI_ENT,
    CAST(CD_RES_ORC AS INT) AS CD_RES_ORC,
    CAST(TX_RES_ORC AS STRING) AS TX_RES_ORC,
    CAST(CD_FAIXA_ORC AS INT) AS CD_FAIXA_ORC,
    CAST(TX_STS_RES AS STRING) AS TX_STS_RES,
    CAST(TX_STS_FINAL AS STRING) AS TX_STS_FINAL,
    CAST(PC_SAI_IND AS DECIMAL(9,6)) AS PC_SAI_IND,
    CAST(PC_SAI_ESS AS DECIMAL(9,6)) AS PC_SAI_ESS,
    CAST(PC_SAI_NAO_ESS AS DECIMAL(9,6)) AS PC_SAI_NAO_ESS,
    CAST(PC_SAI_FUT AS DECIMAL(9,6)) AS PC_SAI_FUT,
    CAST(PC_SAI_OBR AS DECIMAL(9,6)) AS PC_SAI_OBR,
    CAST(PC_REF_IND AS DECIMAL(9,6)) AS PC_REF_IND,
    CAST(PC_REF_ESS AS DECIMAL(9,6)) AS PC_REF_ESS,
    CAST(PC_REF_NAO_ESS AS DECIMAL(9,6)) AS PC_REF_NAO_ESS,
    CAST(PC_REF_FUT AS DECIMAL(9,6)) AS PC_REF_FUT,
    CAST(PC_REF_OBR AS DECIMAL(9,6)) AS PC_REF_OBR,
    CAST(NR_PONT_CONC_IND AS INT) AS NR_PONT_CONC_IND,
    CAST(NR_PONT_CONC_ESS AS INT) AS NR_PONT_CONC_ESS,
    CAST(NR_PONT_CONC_NAO_ESS AS INT) AS NR_PONT_CONC_NAO_ESS,
    CAST(NR_PONT_CONC_FUT AS INT) AS NR_PONT_CONC_FUT,
    CAST(NR_PONT_CONC_OBR AS INT) AS NR_PONT_CONC_OBR,
    CAST(NR_PONT_ORC_IND AS INT) AS NR_PONT_ORC_IND,
    CAST(NR_PONT_ORC_ESS AS INT) AS NR_PONT_ORC_ESS,
    CAST(NR_PONT_ORC_NAO_ESS AS INT) AS NR_PONT_ORC_NAO_ESS,
    CAST(NR_PONT_ORC_FUT AS INT) AS NR_PONT_ORC_FUT,
    CAST(NR_PONT_ORC_OBR AS INT) AS NR_PONT_ORC_OBR,
    CAST(NR_PONT_PRFL_IND AS INT) AS NR_PONT_PRFL_IND,
    CAST(NR_PONT_PRFL_ESS AS INT) AS NR_PONT_PRFL_ESS,
    CAST(NR_PONT_PRFL_NAO_ESS AS INT) AS NR_PONT_PRFL_NAO_ESS,
    CAST(NR_PONT_PRFL_FUT AS INT) AS NR_PONT_PRFL_FUT,
    CAST(NR_PONT_PRFL_OBR AS INT) AS NR_PONT_PRFL_OBR,
    CAST(NR_PONT_IND_FIM AS INT) AS NR_PONT_IND_FIM,
    CAST(NR_PONT_ESS_FIM AS INT) AS NR_PONT_ESS_FIM,
    CAST(NR_PONT_NAO_ESS_FIM AS INT) AS NR_PONT_NAO_ESS_FIM,
    CAST(NR_PONT_FUT_FIM AS INT) AS NR_PONT_FUT_FIM,
    CAST(NR_PONT_OBR_FIM AS INT) AS NR_PONT_OBR_FIM,
    CAST(CD_TEMA_VENCEDOR AS INT) AS CD_TEMA_VENCEDOR,
    CAST(TX_TEMA_VENCEDOR AS STRING) AS TX_TEMA_VENCEDOR,
    CAST(FL_TEM_MOV_AGRO AS STRING) AS FL_TEM_MOV_AGRO,
    CAST(FL_MULT_MOE AS STRING) AS FL_MULT_MOE,
    CAST(FL_PARTICIPA_RADAR AS STRING) AS FL_PARTICIPA_RADAR,
    CAST(DATE('{dt_mes_exea}') AS DATE) AS DT_MES_EXEA
FROM vw_bloco_07
"""
df_ana_edu_fin_cli = spark.sql(query_bloco_08).persist(StorageLevel.MEMORY_AND_DISK)
df_ana_edu_fin_cli.createOrReplaceTempView("vw_ana_edu_fin_cli")

id_final = f"SPARK_FINAL_{dt_mes_exea}"
inicio_final = registrar_etapa(id_final, "inicio")
qt_final = df_ana_edu_fin_cli.count()
registrar_etapa(id_final, "fim", inicio=inicio_final, linhas=qt_final, particoes=df_ana_edu_fin_cli.rdd.getNumPartitions())
print(f"Tabela final materializada via Spark SQL: {qt_final:,} clientes com 72 colunas (vw_ana_edu_fin_cli).")


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Saída final — metadados sem ação adicional
try:
    print("\n[PERF-DIAG] DATAFRAME FINAL")
    print("partições:", df_ana_edu_fin_cli.rdd.getNumPartitions())
    print("storageLevel:", df_ana_edu_fin_cli.storageLevel)
    print("view cacheada:", spark.catalog.isCached("vw_ana_edu_fin_cli"))
    print("colunas:", len(df_ana_edu_fin_cli.columns))
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


## 12. Critérios bloqueantes de qualidade (Spark SQL)

Verificação de unicidade da chave `CD_CLI`, consistência do número de linhas e diagnóstico de clientes sem movimento via Spark SQL.


In [ ]:
%%spark

# Gate explícito: mesmo que o ambiente continue para células posteriores após uma exceção,
# DDL/carga não executam sem todas as validações aprovadas.
VALIDACOES_APROVADAS = False

id_chave = "VALIDA_CHAVE_FINAL"
inicio_chave = registrar_etapa(id_chave, "inicio")
duplicidades = spark.sql("""
    SELECT CD_CLI, COUNT(1) AS QT_DUP
    FROM vw_ana_edu_fin_cli
    GROUP BY CD_CLI
    HAVING COUNT(1) > 1
    LIMIT 1
""").collect()
registrar_etapa(id_chave, "fim", inicio=inicio_chave, linhas=len(duplicidades))

if duplicidades:
    raise ValueError(f"CD_CLI duplicado na saída final: {duplicidades[0]}")
if qt_final != qt_publico_janela:
    raise ValueError(f"Cardinalidade divergente: final={qt_final} != publico={qt_publico_janela}")
if len(df_ana_edu_fin_cli.columns) != 72:
    raise ValueError(f"Schema final deveria ter 72 colunas, mas possui {len(df_ana_edu_fin_cli.columns)}")

id_sem_mov = "CONTA_CLIENTES_SEM_MOVIMENTO"
inicio_sem_mov = registrar_etapa(id_sem_mov, "inicio")
resumo_mov = spark.sql("""
    SELECT
        SUM(CASE WHEN QT_TRANS_TOTAL = 0 THEN 1 ELSE 0 END) AS QT_SEM_MOV,
        SUM(CASE WHEN QT_TRANS_TOTAL > 0 THEN 1 ELSE 0 END) AS QT_COM_MOV
    FROM vw_ana_edu_fin_cli
""").first()
registrar_etapa(id_sem_mov, "fim", inicio=inicio_sem_mov, detalhes=resumo_mov.asDict())

if int(resumo_mov["QT_SEM_MOV"] or 0) + int(resumo_mov["QT_COM_MOV"] or 0) != qt_final:
    raise ValueError("Particionamento clientes com/sem movimento não fecha com qt_final.")

print("Critérios bloqueantes de chave, cardinalidade e movimento aprovados:", resumo_mov.asDict())


## 13. Testes de calendário (Viradas de Mês e Clamping em Spark SQL)


In [ ]:
%%spark

from pyspark.sql.types import StructType, StructField, StringType, IntegerType

id_calendario = "TESTES_CALENDARIO"
inicio_cal = registrar_etapa(id_calendario, "inicio")

casos_calendario = [
    ("dia_10_periodo_1", 10, "2026-08-13 18:42:10", 1, "2026-07-10", "2026-08-09"),
    ("dia_10_periodo_2", 10, "2026-08-13 18:42:10", 2, "2026-06-10", "2026-08-09"),
    ("dia_31_julho", 31, "2026-08-13 09:15:03", 2, "2026-05-31", "2026-07-30"),
    ("fallback_999", 999, "2026-08-12 14:20:00", 2, "2026-06-01", "2026-07-31"),
]

schema_teste_cal = StructType([
    StructField("CASO", StringType(), False),
    StructField("DD_INC", IntegerType(), False),
    StructField("TS_ATL", StringType(), False),
    StructField("PERIODO", IntegerType(), False),
    StructField("DT_INI_ESP", StringType(), False),
    StructField("DT_FIM_ESP", StringType(), False),
])

df_teste_cal = spark.createDataFrame(casos_calendario, schema=schema_teste_cal)
df_teste_cal.createOrReplaceTempView("vw_teste_cal_input")

query_teste_cal = """
WITH base_cal AS (
    SELECT
        *,
        CAST(CASE WHEN DD_INC BETWEEN 1 AND 31 THEN DD_INC ELSE 1 END AS INT) AS DD_CICLO,
        TRUNC(TO_DATE(TS_ATL), 'month') AS DT_MES_REF
    FROM vw_teste_cal_input
),
ciclo_ref AS (
    SELECT
        *,
        DATE_ADD(
            DT_MES_REF,
            CAST(LEAST(DD_CICLO, DAYOFMONTH(LAST_DAY(DT_MES_REF))) - 1 AS INT)
        ) AS DT_INI_CICLO_REF
    FROM base_cal
),
ciclo_aberto AS (
    SELECT
        *,
        CASE
            WHEN TO_DATE(TS_ATL) >= DT_INI_CICLO_REF THEN DT_INI_CICLO_REF
            ELSE DATE_ADD(
                ADD_MONTHS(DT_MES_REF, -1),
                CAST(LEAST(DD_CICLO, DAYOFMONTH(LAST_DAY(ADD_MONTHS(DT_MES_REF, -1)))) - 1 AS INT)
            )
        END AS DT_INI_ABERTO
    FROM ciclo_ref
),
janela_calculada AS (
    SELECT
        *,
        DATE_ADD(
            ADD_MONTHS(TRUNC(DT_INI_ABERTO, 'month'), -PERIODO),
            CAST(
                LEAST(
                    DD_CICLO,
                    DAYOFMONTH(LAST_DAY(ADD_MONTHS(TRUNC(DT_INI_ABERTO, 'month'), -PERIODO)))
                ) - 1 AS INT
            )
        ) AS DT_REF_INI_CALC,
        DATE_SUB(DT_INI_ABERTO, 1) AS DT_REF_FIM_CALC
    FROM ciclo_aberto
)
SELECT
    *,
    (
        CAST(DT_REF_INI_CALC AS STRING) = DT_INI_ESP
        AND CAST(DT_REF_FIM_CALC AS STRING) = DT_FIM_ESP
    ) AS OK
FROM janela_calculada
"""

df_teste_cal_res = spark.sql(query_teste_cal)
falhas_cal = df_teste_cal_res.where("OK = false OR OK IS NULL").collect()
registrar_etapa(id_calendario, "fim", inicio=inicio_cal, linhas=len(falhas_cal))

if falhas_cal:
    raise AssertionError(f"Falha nos testes de calendário: {falhas_cal}")

VALIDACOES_APROVADAS = True
print("Testes de calendário aprovados. Gate global de validações liberado para DDL/carga.")


## 14. DDL particionado de desenvolvimento (72 Colunas Físicas)


In [ ]:
%%spark

if not globals().get("VALIDACOES_APROVADAS", False):
    raise RuntimeError("DDL bloqueado: validações de qualidade/calendário não foram aprovadas nesta sessão.")

ddl_tabela_spark = f"""
CREATE TABLE IF NOT EXISTS {tabela_spark} (
    CD_CLI                     INT             COMMENT 'Código do cliente',
    TS_ATL_TRAN                TIMESTAMP       COMMENT 'Maior timestamp de atualização das transações do cliente no recorte definido por qtd_mes_pbco_alvo',
    DD_INC_MM_CLC_BLC          SMALLINT        COMMENT 'Dia inicial do cálculo do balanço: 1 a 31; 996 = múltiplas contas elegíveis; 997 = sem conta BB corrente identificável; 999 = conta sem data cadastrada',
    VL_REN_PRES                DECIMAL(18,2)   COMMENT 'Renda presumida de RDPR_PF: valor >= 0 é saída válida; -1 sem escoragem; -2 CPF inválido; -3 não PF; -4 identidade ambígua; -5 duplicidade do modelo; -6 renda nula; -7 renda negativa na origem',
    CD_MAC_PRFL_CLI            BIGINT          COMMENT 'Código do macroperfil financeiro',
    NM_MAC_PRFL_CLI            STRING          COMMENT 'Texto do macroperfil financeiro; CODIGO NAO MAPEADO quando o código estiver fora do domínio conhecido',
    CD_MIC_PRFL_CLI            BIGINT          COMMENT 'Código do microperfil financeiro',
    NM_MIC_PRFL_CLI            STRING          COMMENT 'Texto do microperfil financeiro; CODIGO NAO MAPEADO quando o código estiver fora do domínio conhecido',
    NM_PRFL_FIN                STRING          COMMENT 'Texto unificado do perfil financeiro; nulo sem código e CODIGO NAO MAPEADO para código fora do domínio conhecido',
    DT_REF_INI                 DATE            COMMENT 'Menor data inicial entre os ciclos financeiros fechados selecionados',
    DT_REF_FIM                 DATE            COMMENT 'Maior data final entre os ciclos financeiros fechados selecionados',
    DT_EXEA                    DATE            COMMENT 'Data de execução do ETL',
    QT_TRANS_TOTAL             BIGINT          COMMENT 'Quantidade total de transações',
    QT_TRANS_ENT               BIGINT          COMMENT 'Quantidade de transações de entrada',
    QT_TRANS_SAI               BIGINT          COMMENT 'Quantidade de transações de saída',
    VL_TRANS_ENT               DECIMAL(25,2)   COMMENT 'Valor total de entradas participantes (_IN_PARTICIPA = 1)',
    VL_TRANS_SAI               DECIMAL(25,2)   COMMENT 'Valor total de saídas participantes (_IN_PARTICIPA = 1)',
    VL_ENT_REN                 DECIMAL(18,2)   COMMENT 'Valores recebidos que representam renda, remuneração ou benefícios',
    VL_ENT_EST                 DECIMAL(18,2)   COMMENT 'Valores devolvidos ou recebidos de volta por correções, cancelamentos ou ajustes',
    VL_ENT_RESG                DECIMAL(18,2)   COMMENT 'Valores recuperados de investimentos ou aplicações financeiras',
    VL_ENT_OUT                 DECIMAL(18,2)   COMMENT 'Valores recebidos cuja origem não foi identificada ou não se enquadram nas demais classificações',
    VL_ENT_CRED                DECIMAL(18,2)   COMMENT 'Valores obtidos por empréstimos, financiamentos ou outras operações de crédito',
    VL_ENT_TOTAL               DECIMAL(18,2)   COMMENT 'Valor total de entrada participante',
    VL_SAI_IND                 DECIMAL(18,2)   COMMENT 'Saídas cujo destino não foi identificado ou não se enquadram nas demais classificações',
    VL_SAI_ESS                 DECIMAL(18,2)   COMMENT 'Gastos necessários para a manutenção da vida e do dia a dia',
    VL_SAI_NAO_ESS                DECIMAL(18,2)   COMMENT 'Gastos relacionados a escolhas pessoais, lazer e estilo de vida',
    VL_SAI_FUT                 DECIMAL(18,2)   COMMENT 'Valores destinados à formação de patrimônio, reserva ou objetivos futuros',
    VL_SAI_OBR                 DECIMAL(18,2)   COMMENT 'Valores destinados ao pagamento de dívidas, parcelas e compromissos financeiros',
    VL_SAI_TOTAL               DECIMAL(18,2)   COMMENT 'Valor total de saída participante',
    VL_RES_ORC                 DECIMAL(18,2)   COMMENT 'Valor do resultado do orçamento: entradas menos saídas',
    PC_SAI_ENT                 DECIMAL(9,6)    COMMENT 'Percentual do valor total de saídas sobre o valor total de entradas: VL_SAI_TOTAL / VL_ENT_TOTAL',
    CD_RES_ORC                 INT             COMMENT 'Código do resultado do orçamento: 0 = Neutro; 1 = Superavitário; 2 = Deficitário',
    TX_RES_ORC                 STRING          COMMENT 'Texto do resultado do orçamento',
    CD_FAIXA_ORC               INT             COMMENT 'Código da faixa do resultado orçamentário: 0 = Neutro; 1 = Deficitário Moderado; 2 = Deficitário Acentuado; 3 = Superavitário Moderado; 4 = Superavitário Acentuado',
    TX_STS_RES                 STRING          COMMENT 'Status da intensidade do resultado: Acentuado ou Moderado; nulo quando o resultado for Neutro',
    TX_STS_FINAL               STRING          COMMENT 'Texto final composto pelo resultado e seu status',
    PC_SAI_IND                 DECIMAL(9,6)    COMMENT 'Percentual do valor de saída indeterminada sobre a renda presumida: VL_SAI_IND / VL_REN_PRES',
    PC_SAI_ESS                 DECIMAL(9,6)    COMMENT 'Percentual do valor de saída essencial sobre a renda presumida: VL_SAI_ESS / VL_REN_PRES',
    PC_SAI_NAO_ESS                DECIMAL(9,6)    COMMENT 'Percentual do valor de saída não essencial sobre a renda presumida: VL_SAI_NAO_ESS / VL_REN_PRES',
    PC_SAI_FUT                 DECIMAL(9,6)    COMMENT 'Percentual do valor de saída para o futuro sobre a renda presumida: VL_SAI_FUT / VL_REN_PRES',
    PC_SAI_OBR                 DECIMAL(9,6)    COMMENT 'Percentual do valor de saída para obrigações sobre a renda presumida: VL_SAI_OBR / VL_REN_PRES',
    PC_REF_IND                 DECIMAL(9,6)    COMMENT 'Percentual de referência para saída indeterminada',
    PC_REF_ESS                 DECIMAL(9,6)    COMMENT 'Percentual de referência para saída essencial',
    PC_REF_NAO_ESS                DECIMAL(9,6)    COMMENT 'Percentual de referência para saída não essencial',
    PC_REF_FUT                 DECIMAL(9,6)    COMMENT 'Percentual de referência para saída destinada ao futuro',
    PC_REF_OBR                 DECIMAL(9,6)    COMMENT 'Percentual de referência para saída de obrigações',
    NR_PONT_CONC_IND           INT             COMMENT 'Pontuação de concentração da saída indeterminada',
    NR_PONT_CONC_ESS           INT             COMMENT 'Pontuação de concentração essencial',
    NR_PONT_CONC_NAO_ESS          INT             COMMENT 'Pontuação de concentração não essencial',
    NR_PONT_CONC_FUT           INT             COMMENT 'Pontuação de concentração da saída destinada ao futuro',
    NR_PONT_CONC_OBR           INT             COMMENT 'Pontuação de concentração da saída de obrigações',
    NR_PONT_ORC_IND            INT             COMMENT 'Pontuação orçamentária da classificação indeterminada',
    NR_PONT_ORC_ESS            INT             COMMENT 'Pontuação de orçamento essencial',
    NR_PONT_ORC_NAO_ESS           INT             COMMENT 'Pontuação de orçamento não essencial',
    NR_PONT_ORC_FUT            INT             COMMENT 'Pontuação orçamentária da classificação futuro',
    NR_PONT_ORC_OBR            INT             COMMENT 'Pontuação orçamentária da classificação obrigações',
    NR_PONT_PRFL_IND           INT             COMMENT 'Pontuação por perfil da classificação indeterminada',
    NR_PONT_PRFL_ESS           INT             COMMENT 'Pontuação de perfil essencial',
    NR_PONT_PRFL_NAO_ESS          INT             COMMENT 'Pontuação de perfil não essencial',
    NR_PONT_PRFL_FUT           INT             COMMENT 'Pontuação por perfil da classificação futuro',
    NR_PONT_PRFL_OBR           INT             COMMENT 'Pontuação por perfil da classificação obrigações',
    NR_PONT_IND_FIM            INT             COMMENT 'Pontuação final da classificação indeterminada',
    NR_PONT_ESS_FIM            INT             COMMENT 'Pontuação final da classificação essenciais',
    NR_PONT_NAO_ESS_FIM           INT             COMMENT 'Pontuação final da classificação não essenciais',
    NR_PONT_FUT_FIM            INT             COMMENT 'Pontuação final da classificação futuro',
    NR_PONT_OBR_FIM            INT             COMMENT 'Pontuação final da classificação obrigações',
    CD_TEMA_VENCEDOR           INT             COMMENT 'Código do conceito vencedor: 1 = Categorização dos Gastos; 2 = Gestão de Orçamento; 3 = Consumo Planejado; 4 = Formação de Reserva; 5 = Uso Consciente do Crédito; 9 = Empate; nulo sem pontuação completa',
    TX_TEMA_VENCEDOR           STRING          COMMENT 'Texto do conceito vencedor; nulo quando as cinco pontuações finais não estiverem preenchidas',
    FL_TEM_MOV_AGRO            STRING          COMMENT 'Indica se o cliente teve movimentação de crédito ou débito em categoria marcada como agro: S ou N',
    FL_MULT_MOE                STRING          COMMENT 'Indica se o cliente teve movimentação em mais de um tipo de moeda ou moeda diferente de BRL na janela: S ou N',
    FL_PARTICIPA_RADAR         STRING          COMMENT 'Campo reservado para regra futura de participação; nulo nesta versão'
)
PARTITIONED BY (
    DT_MES_EXEA DATE COMMENT 'Mês de execução do ETL, representado pelo primeiro dia do mês'
)
COMMENT 'Análise de Educação Financeira do Cliente (Metadata-Driven & 100% Spark SQL)'
STORED AS PARQUET
"""

if RECRIAR_TABELA_FINAL:
    spark.sql(f"DROP TABLE IF EXISTS {tabela_spark}")
    print(f"Tabela anterior removida: {tabela_spark}")

spark.sql(ddl_tabela_spark)

schema_df_final = {f.name: f.dataType.simpleString().lower() for f in df_ana_edu_fin_cli.schema.fields}
schema_hive = {f.name: f.dataType.simpleString().lower() for f in spark.table(tabela_spark).schema.fields}

if list(schema_df_final.keys()) != list(schema_hive.keys()):
    raise ValueError(
        "Ordem/nome das colunas diverge entre DataFrame final e tabela Hive. "
        f"DF={list(schema_df_final.keys())} | HIVE={list(schema_hive.keys())}"
    )

divergencias_tipo = {
    nome: (schema_df_final[nome], schema_hive[nome])
    for nome in schema_df_final
    if schema_df_final[nome] != schema_hive[nome]
}
if divergencias_tipo:
    raise ValueError(f"Tipos divergentes entre DataFrame e Hive: {divergencias_tipo}")

print(f"Tabela Hive garantida e contrato de 72 colunas validado: {tabela_spark}")


## 15. Carga Hive/Parquet idempotente no sandbox

A V15 não usa `saveAsTable`. A competência corrente é gravada por `INSERT OVERWRITE` estático na partição `DT_MES_EXEA`, seguida por leitura de validação da própria tabela. A carga só é registrada como concluída quando `COUNT(Hive) == qt_final`.


In [ ]:
%%spark

if not globals().get("VALIDACOES_APROVADAS", False):
    raise RuntimeError("Carga bloqueada: validações de qualidade/calendário não foram aprovadas nesta sessão.")

id_carga = f"CARGA_OVERWRITE_PARTICAO_{dt_mes_exea}"
inicio_carga = registrar_etapa(id_carga, "inicio")

# O DDL define DT_MES_EXEA como partição e todas as demais colunas na ordem física da tabela.
colunas_hive = [f.name for f in spark.table(tabela_spark).schema.fields]
colunas_dados_hive = [c for c in colunas_hive if c != "DT_MES_EXEA"]

if set(colunas_dados_hive) != set(c for c in df_ana_edu_fin_cli.columns if c != "DT_MES_EXEA"):
    raise ValueError("Conjunto de colunas do DataFrame não coincide com as colunas de dados da tabela Hive.")

# Reduz arquivos pequenos na partição sem shuffle adicional.
df_carga_hive = df_ana_edu_fin_cli.select(*colunas_dados_hive).coalesce(QTD_PARTICOES_ESCRITA)
df_carga_hive.createOrReplaceTempView("vw_carga_ana_edu_fin_cli")

select_carga = ",\n        ".join(f"`{nome}`" for nome in colunas_dados_hive)

sql_carga = f"""
INSERT OVERWRITE TABLE {tabela_spark}
PARTITION (DT_MES_EXEA = '{dt_mes_exea}')
SELECT
        {select_carga}
FROM vw_carga_ana_edu_fin_cli
"""

try:
    spark.sql(sql_carga)
except Exception as exc:
    print(f"Erro no INSERT OVERWRITE Hive da competência {dt_mes_exea}: {exc}")
    raise

# Prova pós-carga: consulta a própria partição persistida.
inicio_valida_carga = registrar_etapa(f"VALIDA_CARGA_{dt_mes_exea}", "inicio")
qt_hive = int(spark.sql(f"""
    SELECT COUNT(1) AS QT
    FROM {tabela_spark}
    WHERE DT_MES_EXEA = DATE('{dt_mes_exea}')
""").first()["QT"])
registrar_etapa(
    f"VALIDA_CARGA_{dt_mes_exea}",
    "fim",
    inicio=inicio_valida_carga,
    linhas=qt_hive,
)

if qt_hive != qt_final:
    raise AssertionError(
        f"Carga inválida na partição {dt_mes_exea}: Hive={qt_hive} != DataFrame={qt_final}"
    )

registrar_etapa(
    id_carga,
    "fim",
    inicio=inicio_carga,
    linhas=qt_hive,
    particoes=QTD_PARTICOES_ESCRITA,
    detalhes={"qt_dataframe": qt_final, "qt_hive": qt_hive, "modo": "INSERT_OVERWRITE_PARTICAO"},
)

print("=" * 80)
print("CARGA HIVE VALIDADA COM SUCESSO")
print(f"Tabela       : {tabela_spark}")
print(f"Competência  : {dt_mes_exea}")
print(f"DataFrame    : {qt_final:,} linhas")
print(f"Hive         : {qt_hive:,} linhas")
print(f"Writers alvo : {QTD_PARTICOES_ESCRITA}")
print("Status       : OK")
print("=" * 80)


In [ ]:
%%spark
# [PERF-DIAG][REMOVER] Pós-carga — usa resultados já produzidos
try:
    print("\n[PERF-DIAG] PÓS-CARGA")
    print("tabela:", tabela_spark)
    print("competência:", dt_mes_exea)
    print("linhas DataFrame:", qt_final)
    print("linhas Hive:", qt_hive)
    print("partições alvo de escrita:", QTD_PARTICOES_ESCRITA)
    print("status:", "OK" if qt_hive == qt_final else "DIVERGENTE")
except Exception as exc:
    print(f"[PERF-DIAG] AVISO: {type(exc).__name__}: {exc} | execução funcional não afetada.")


## 16. Liberação final de caches e sumário de etapas


In [ ]:
%%spark

for nome_df in (
    "df_publico_janela",
    "df_perfil_financeiro",
    "df_renda_presumida",
    "df_acumulador",
    "df_ana_edu_fin_cli",
):
    obj = globals().get(nome_df)
    if obj is not None:
        try:
            obj.unpersist()
        except Exception:
            pass

print("Todos os DataFrames principais em cache foram liberados.")
print("\n" + "="*92)
print(f"{'SUMÁRIO DE EXECUÇÃO DO PIPELINE V15':^92}")
print("="*92)
for r in historico_etapas:
    linhas_txt = "-" if r["linhas"] is None else str(r["linhas"])
    particoes_txt = "-" if r["particoes"] is None else str(r["particoes"])
    print(f"- {r['etapa']:<48} | Duração: {r['duracao_segundos']:>8.2f}s | Linhas: {linhas_txt:>10} | Partições: {particoes_txt:>4}")
print("="*92)
